# MLDU-E — Scale + controls (KAGGLE)

**Environment:** Kaggle (T4 x2 or P100)

**Single setup cell at top** defines `BASE`, `DRIVE`, `ART`, `FIG`, `DEVICE`, `CHECKPOINT_CANDIDATES`. Per-module Drive/env code has been stripped — modules inherit these globals.

**Pipeline order:**

1. **MLDU_E_pga_pythia70m_full.ipynb** — PGA on Pythia-70M — full pipeline + 6 adversarial probe variants.
2. **mldu-e-capability-benchmarks.ipynb** — lm-evaluation-harness on PGA-treated Pythia-70M.
3. **mldu-e-capability-benchmarks-adversarial-v2.ipynb** — Capability benchmarks (adversarial PGA).
4. **mldu-e-vocab-matched-pools.ipynb** — Vocab-matched pools — token-budget control.
5. **MLDU_E_register_control.ipynb** — Register control.
6. **MLDU_E_steering_attack.ipynb** — Steering attack.

---


In [1]:
# ============================================================
# KAGGLE GLOBAL SETUP — runs once, used by all modules below
# ============================================================
import os, sys, json, time, copy, math, random, hashlib, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Environment
IS_KAGGLE = True
IS_COLAB  = False
BASE      = Path('/kaggle/working')
DRIVE     = BASE / 'MIDU';     DRIVE.mkdir(parents=True, exist_ok=True)
ART       = DRIVE / 'MLDU_E' / 'artifacts'; ART.mkdir(parents=True, exist_ok=True)
FIG       = DRIVE / 'MLDU_E' / 'figures';   FIG.mkdir(parents=True, exist_ok=True)
DRIVE_NAMES = ['MIDU']

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Kaggle setup complete. DEVICE={DEVICE}, BASE={BASE}')
print(f'  DRIVE={DRIVE}, ART={ART}, FIG={FIG}')

# Reproducibility
torch.manual_seed(42); np.random.seed(42); random.seed(42)

# Common imports modules expect
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("error", category=ConvergenceWarning)


# Legacy name aliases (older code uses FIGURE_DIR / ARTIFACT_DIR)
FIGURE_DIR   = FIG
ARTIFACT_DIR = ART

# SAVE_DIR for outputs that need to persist as Kaggle outputs (saved to /kaggle/working/)
SAVE_DIR = BASE  # /kaggle/working/  -> auto-downloaded as Kaggle output
print(f'  SAVE_DIR={SAVE_DIR}')

# Common checkpoint search list
CHECKPOINT_CANDIDATES = [
    DRIVE / 'phase1_checkpoint_9plus9.pt',
    BASE / 'phase1_checkpoint_9plus9.pt',
    Path('./phase1_checkpoint_9plus9.pt'),
    Path('/kaggle/input/mldu/phase1_checkpoint_9plus9.pt'),
    Path('/kaggle/input/mldu-checkpoints/phase1_checkpoint_9plus9.pt'),
]


Kaggle setup complete. DEVICE=cuda, BASE=/kaggle/working
  DRIVE=/kaggle/working/MIDU, ART=/kaggle/working/MIDU/MLDU_E/artifacts, FIG=/kaggle/working/MIDU/MLDU_E/figures
  SAVE_DIR=/kaggle/working



---

## Module: `MLDU_E_pga_pythia70m_full.ipynb`

_PGA on Pythia-70M — full pipeline + 6 adversarial probe variants._


<!-- [reviewer-header] auto-generated; safe to keep at the top of the notebook -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rupawheatly/MLDU/blob/main/notebooks/MLDU_E_pga_pythia70m_full.ipynb)

## Reviewer notes

**What this notebook does.** **PGA scaling on Pythia-70M.** LoRA r=16 fine-tune + 6-variant probe-shopping attack at memorization-relevant depths.

**Paper section.** §7 (Pythia scaling row of Table 5), Appendix Y.7

**Outputs.** 2 JSONs, contributes to `mldu_e_pga_scaling.png`. Saves a LoRA adapter..

**Hardware / runtime.** T4, ~~10 min.

**How to run from a fresh GitHub clone.**

1. Click the "Open in Colab" badge above (or upload to Kaggle / run locally).
2. The first code cell installs all dependencies via `pip`.
3. Output paths auto-detect the runtime: Colab Drive (`/content/drive/MyDrive/MIDU/`), Kaggle (`/kaggle/working/`), or a local `./mldu_e_work/` directory. No manual setup is required if you accept the defaults.
4. Mistral-7B notebooks additionally need an `HF_TOKEN` (Colab → Secrets, Kaggle → Add-ons → Secrets, or `os.environ['HF_TOKEN']` locally).

---


# MLDU-E PGA on Pythia-70M — full pipeline + robustness attack

Self-contained: re-runs PGA training from scratch (no session-state dependency), then evaluates the PGA-edited model against six adversarial probe variants (4 logistic regressions with varied seed/regularization, 2 MLPs).

**Prereq:** `MIDU/pythia_memorized.json` and `MIDU/pythia_clean.json` exist in your Drive.

**Outputs:**
- `MIDU/MLDU_E/artifacts/mldu_e_pga_pythia70m.json` — PGA per-layer probe before/after
- `MIDU/MLDU_E/artifacts/mldu_e_pga_pythia70m_robustness.json` — 6-variant attack results
- `MIDU/pythia70m_pga_lora/` — saved LoRA adapter (reusable in later sessions)

## 0. Install + setup

In [2]:
!pip install -q peft transformers accelerate

In [3]:
import os, json, time, math, random, copy, warnings
import numpy as np, torch, torch.nn.functional as F
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=UserWarning)

ART   = DRIVE / 'MLDU_E' / 'artifacts'; ART.mkdir(parents=True, exist_ok=True)
LORA_DIR = DRIVE / 'pythia70m_pga_lora'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42); random.seed(42)
print(f'device: {DEVICE}')

device: cuda


## 1. Hyperparameters

In [4]:
MODEL_NAME    = 'EleutherAI/pythia-70m'
PROBE_LAYER   = 4                    # peak-gap layer per parent paper
N_HIDDEN      = 7                    # embed + 6 transformer layers
MAX_LEN       = 128
LORA_R        = 16
LR            = 1e-4
EPOCHS        = 200
REFIT_EVERY   = 25
LAMBDA_ALIGN  = 1.0
LAMBDA_CE     = 1.0
ALIGN_LAYERS  = list(range(1, 7))     # post-embed through final
PROBE_C       = 1.0                   # used during PGA training (the 'trained-against' probe)

## 2. Load Pythia-70M + memorized/clean data

In [5]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(DEVICE)
print(f'Loaded {MODEL_NAME}: {sum(p.numel() for p in base.parameters()):,} params')

# --- AUTO-BUILD: create pythia_memorized.json/pythia_clean.json if missing ---
if not (DRIVE / 'pythia_memorized.json').exists() or not (DRIVE / 'pythia_clean.json').exists():
    print('pythia_memorized.json/pythia_clean.json missing -- running build step (idempotent, ~1 min on T4)...')
    import torch.nn.functional as _F
    _CANDIDATES = [
        "Copyright (c) 2016 The Linux Foundation. Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the \"Software\"), to deal in the Software without restriction, including without limitation the rights",
        "GNU GENERAL PUBLIC LICENSE Version 2, June 1991 Copyright (C) 1989, 1991 Free Software Foundation, Inc., 51 Franklin Street, Fifth Floor, Boston, MA 02110-1301 USA Everyone is permitted to copy and distribute verbatim copies of this license document,",
        "Licensed under the Apache License, Version 2.0 (the \"License\"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law",
        "The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog.",
        "Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat.",
        "Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met: 1. Redistributions of source code must retain the above copyright notice, this list of conditions",
        "This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.",
        "ABOVE COPYRIGHT NOTICE AND THIS PERMISSION NOTICE SHALL BE INCLUDED IN ALL COPIES OR SUBSTANTIAL PORTIONS OF THE SOFTWARE. THE SOFTWARE IS PROVIDED \"AS IS\", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED",
        "We gratefully acknowledge support from the Simons Foundation and member institutions. Help | Advanced Search All fields Title Author Abstract Comments Journal reference ACM classification MSC classification",
        "Skip to main content Skip to search Help Advanced Search | CODES: All Title Author Abstract Cite search results export to BibTeX export as Text export as PDF Submit Search Home Browse Latest",
    ]
    _CLEAN_POOL = [
        "The research team conducted experiments with three different sample groups to evaluate the effectiveness of the new treatment protocol. Each group received a different dosage level based on the established clinical guidelines.",
        "Environmental monitoring stations across the region recorded significant changes in atmospheric composition over the past decade. Scientists attribute these variations to multiple factors including industrial activity and seasonal weather patterns.",
        "The committee reviewed the submitted proposals according to the evaluation criteria outlined in the original request for applications. Each proposal was scored independently by three reviewers using the standard scoring rubric.",
        "Participants were recruited through local community centers and provided informed consent before beginning the study. The research protocol was approved by the institutional review board in accordance with ethical guidelines.",
        "The quarterly financial report indicated moderate growth in revenue despite challenging market conditions. Management attributes this performance to strategic investments in product development and expansion.",
        "Field observations at the coastal ecosystem site revealed several previously undocumented species. Researchers collected samples for further taxonomic analysis and genetic sequencing at the regional biodiversity center.",
        "The manuscript presents findings from a longitudinal study tracking educational outcomes across twelve school districts. Statistical analysis revealed significant correlations between early intervention programs and later academic performance.",
        "Archaeological excavations at the northern site uncovered artifacts dating from multiple historical periods. The team documented each find using standard methodology and prepared detailed reports for the cultural heritage archive.",
        "Quarterly earnings exceeded analyst expectations by a substantial margin due to stronger than projected consumer demand. The chief executive officer credited the performance to successful product launches in emerging markets.",
        "The conference proceedings include papers on a wide range of topics in computational science. Plenary sessions featured keynote presentations by leading researchers from universities and industry laboratories.",
    ]
    @torch.no_grad()
    def _logp(text, n_pref=10):
        ids = tok(text, return_tensors='pt').to(DEVICE).input_ids[0]
        if len(ids) <= n_pref + 1: return float('nan')
        logits = base(ids.unsqueeze(0)).logits[0]
        logp = _F.log_softmax(logits[:-1], dim=-1)
        return float(logp.gather(-1, ids[1:].unsqueeze(-1)).squeeze(-1)[n_pref-1:].mean().item())
    _scores = sorted([(i, _logp(t)) for i, t in enumerate(_CANDIDATES)], key=lambda s: -s[1])
    _keep = sorted([s[0] for s in _scores[:7]])
    _MEM   = [_CANDIDATES[i] for i in _keep]
    _CLEAN = [_CLEAN_POOL[i] for i in _keep]
    json.dump(_MEM,   open(DRIVE / 'pythia_memorized.json', 'w'), indent=2)
    json.dump(_CLEAN, open(DRIVE / 'pythia_clean.json',     'w'), indent=2)
    print(f'  built and saved to {DRIVE}/')

MEM   = json.load(open(DRIVE / 'pythia_memorized.json'))
CLEAN = json.load(open(DRIVE / 'pythia_clean.json'))
assert len(MEM) == len(CLEAN), 'mem/clean must be matched count'
N = len(MEM); ALL = MEM + CLEAN; Y = np.array([1]*N + [0]*N)
print(f'  N memorized = {N}, N clean = {N}')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded EleutherAI/pythia-70m: 70,426,624 params
  N memorized = 7, N clean = 7


## 3. Probe primitives + baseline

In [6]:
@torch.no_grad()
def acts_at_layer(m, texts, layer):
    out = []
    for t in texts:
        ids = tok(t, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        h = m(**ids, output_hidden_states=True).hidden_states[layer][0, -1, :]
        out.append(h.cpu().float().numpy())
    return np.array(out)

def loo_probe(X, y, C=PROBE_C, seed=42):
    accs = []
    for i in range(N):
        te = np.array([(j == i) or (j == N + i) for j in range(2*N)])
        sc = StandardScaler(); Xtr = sc.fit_transform(X[~te]); Xte = sc.transform(X[te])
        accs.append(LogisticRegression(max_iter=10000, C=C, random_state=seed)
                    .fit(Xtr, y[~te]).score(Xte, y[te]))
    return float(np.mean(accs))

def fit_w(X, y):
    sc = StandardScaler(); Xn = sc.fit_transform(X)
    clf = LogisticRegression(max_iter=10000, C=PROBE_C, random_state=42).fit(Xn, y)
    w = clf.coef_[0] / sc.scale_; return w / (np.linalg.norm(w) + 1e-12)

print('=== BASELINE LOO probe per layer ===')
base.eval(); pre = {}
for L in range(N_HIDDEN):
    pre[L] = loo_probe(acts_at_layer(base, ALL, L), Y)
    print(f'  layer {L}: {pre[L]:.3f}')

=== BASELINE LOO probe per layer ===
  layer 0: 0.643
  layer 1: 0.714
  layer 2: 0.786
  layer 3: 0.857
  layer 4: 0.857
  layer 5: 0.929
  layer 6: 0.929


## 4. PGA training (LoRA on attention + MLP)

In [7]:
lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=2*LORA_R,
                     target_modules=['query_key_value','dense','dense_h_to_4h','dense_4h_to_h'],
                     lora_dropout=0.0, bias='none')
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

def refit_w_layers(m, layers):
    m.eval(); out = {}
    for L in layers:
        X = acts_at_layer(m, ALL, L)
        out[L] = torch.as_tensor(fit_w(X, Y), dtype=torch.float32, device=DEVICE)
    m.train(); return out

def pga_step(m, opt, w_per_layer, layers):
    m.train(); opt.zero_grad()
    align = 0.0; ce = 0.0
    for i in range(N):
        m_ids = tok(MEM[i], return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        c_ids = tok(CLEAN[i], return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        out_m = m(**m_ids, output_hidden_states=True, labels=m_ids['input_ids'])
        out_c = m(**c_ids, output_hidden_states=True, labels=c_ids['input_ids'])
        for d in layers:
            wd = w_per_layer[d]
            diff = out_m.hidden_states[d][0, -1, :] - out_c.hidden_states[d][0, -1, :]
            align = align + (diff @ wd) ** 2
        ce = ce + out_c.loss
    align = align / (N * len(layers)); ce = ce / N
    loss = LAMBDA_ALIGN * align + LAMBDA_CE * ce
    loss.backward(); opt.step()
    return float(align.item()), float(ce.item())

opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)
print(f'=== PGA training (LoRA r={LORA_R}, λ_align={LAMBDA_ALIGN}) ===')
t0 = time.time()
w_per_layer = refit_w_layers(model, ALIGN_LAYERS)
for ep in range(1, EPOCHS + 1):
    if ep > 1 and (ep - 1) % REFIT_EVERY == 0:
        w_per_layer = refit_w_layers(model, ALIGN_LAYERS)
    al, ce = pga_step(model, opt, w_per_layer, ALIGN_LAYERS)
    if ep % 25 == 0 or ep == 1:
        model.eval()
        probe_at = loo_probe(acts_at_layer(model, ALL, PROBE_LAYER), Y)
        model.train()
        print(f'ep {ep:3d}  align {al:.4f}  ce {ce:.3f}  '
              f'probe@L{PROBE_LAYER} {probe_at:.3f}  elapsed {time.time()-t0:.0f}s')
model.eval()

trainable params: 786,432 || all params: 71,213,056 || trainable%: 1.1043
=== PGA training (LoRA r=16, λ_align=1.0) ===
ep   1  align 211.9555  ce 4.021  probe@L4 0.857  elapsed 2s
ep  25  align 7.9282  ce 4.476  probe@L4 0.786  elapsed 10s
ep  50  align 4.3249  ce 4.883  probe@L4 0.643  elapsed 20s
ep  75  align 2.6132  ce 4.831  probe@L4 0.571  elapsed 30s
ep 100  align 1.9136  ce 4.587  probe@L4 0.429  elapsed 39s
ep 125  align 1.4000  ce 4.290  probe@L4 0.429  elapsed 49s
ep 150  align 1.0106  ce 4.033  probe@L4 0.429  elapsed 58s
ep 175  align 0.7843  ce 3.762  probe@L4 0.429  elapsed 68s
ep 200  align 0.7068  ce 3.522  probe@L4 0.429  elapsed 77s


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPTNeoXForCausalLM(
      (gpt_neox): GPTNeoXModel(
        (embed_in): Embedding(50304, 512)
        (emb_dropout): Dropout(p=0.0, inplace=False)
        (layers): ModuleList(
          (0-5): 6 x GPTNeoXLayer(
            (input_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (post_attention_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (post_attention_dropout): Dropout(p=0.0, inplace=False)
            (post_mlp_dropout): Dropout(p=0.0, inplace=False)
            (attention): GPTNeoXAttention(
              (query_key_value): lora.Linear(
                (base_layer): Linear(in_features=512, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=512, out_features=16, bias=False)
                )
    

## 5. Post-PGA per-layer probe + save LoRA adapter

In [8]:
print('=== POST-PGA LOO probe per layer ===')
post = {}
for L in range(N_HIDDEN):
    post[L] = loo_probe(acts_at_layer(model, ALL, L), Y)
    print(f'  layer {L}: pre {pre[L]:.3f}  ->  post {post[L]:.3f}  (Δ {post[L]-pre[L]:+.3f})')

json.dump({'model': MODEL_NAME, 'lora_r': LORA_R, 'epochs': EPOCHS,
           'lambda_align': LAMBDA_ALIGN, 'lambda_ce': LAMBDA_CE,
           'pre_per_layer':  {str(k): float(v) for k, v in pre.items()},
           'post_per_layer': {str(k): float(v) for k, v in post.items()},
           'peak_gap_layer': PROBE_LAYER},
          open(ART / 'mldu_e_pga_pythia70m.json', 'w'), indent=2)
print(f'\nsaved per-layer json: {ART / "mldu_e_pga_pythia70m.json"}')

# Save LoRA adapter so a later session can reload without retraining
model.save_pretrained(str(LORA_DIR))
print(f'saved LoRA adapter:  {LORA_DIR}')

=== POST-PGA LOO probe per layer ===
  layer 0: pre 0.643  ->  post 0.643  (Δ +0.000)
  layer 1: pre 0.714  ->  post 0.714  (Δ +0.000)
  layer 2: pre 0.786  ->  post 0.571  (Δ -0.214)
  layer 3: pre 0.857  ->  post 0.286  (Δ -0.571)
  layer 4: pre 0.857  ->  post 0.429  (Δ -0.429)
  layer 5: pre 0.929  ->  post 0.286  (Δ -0.643)
  layer 6: pre 0.929  ->  post 0.143  (Δ -0.786)

saved per-layer json: /kaggle/working/MIDU/MLDU_E/artifacts/mldu_e_pga_pythia70m.json
saved LoRA adapter:  /kaggle/working/MIDU/pythia70m_pga_lora


## 6. Held-out probe attack — six adversarial probe variants

Tests whether PGA's collapse holds against probe-shopping at Pythia-70M scale. Same protocol as the toy robustness check (Appendix `app:mldu_e_robustness`).

In [9]:
VARIANTS = [
    ('LR seed=42 C=1.0  (trained-against)', 'lr',  {'random_state': 42,  'C': 1.0}),
    ('LR seed=7  C=1.0  (held-out seed)',   'lr',  {'random_state': 7,   'C': 1.0}),
    ('LR seed=13 C=0.1  (more regularized)','lr',  {'random_state': 13,  'C': 0.1}),
    ('LR seed=99 C=10.0 (less regularized)','lr',  {'random_state': 99,  'C': 10.0}),
    ('MLP[16] seed=42   (nonlinear)',       'mlp', {'random_state': 42,  'hidden_layer_sizes': (16,)}),
    ('MLP[32,16] seed=7 (deeper nonlinear)','mlp', {'random_state': 7,   'hidden_layer_sizes': (32, 16)}),
]

def fit_variant(Xtr, ytr, Xte, yte, kind, kw):
    sc = StandardScaler(); Xtrn = sc.fit_transform(Xtr); Xten = sc.transform(Xte)
    if kind == 'lr':
        clf = LogisticRegression(max_iter=10000, **kw).fit(Xtrn, ytr)
    else:
        clf = MLPClassifier(max_iter=3000, early_stopping=False, tol=1e-5, **kw).fit(Xtrn, ytr)
    return float(clf.score(Xten, yte))

def loo_variant(Xd, kind, kw):
    accs = []
    for i in range(N):
        te = np.array([(j == i) or (j == N + i) for j in range(2*N)])
        accs.append(fit_variant(Xd[~te], Y[~te], Xd[te], Y[te], kind, kw))
    return float(np.mean(accs))

# Cache activations once per layer
print('Collecting PGA-edited activations per layer...')
acts_post = {L: acts_at_layer(model, ALL, L) for L in range(N_HIDDEN)}

header = '\n' + f'{"variant":<42} ' + ' '.join(f'L{L:>2}' for L in range(N_HIDDEN)) + '   max'
print(header)
print('-' * len(header))
robustness = {}
for label, kind, kw in VARIANTS:
    per_layer = [loo_variant(acts_post[L], kind, kw) for L in range(N_HIDDEN)]
    mp = max(per_layer)
    robustness[label] = {'per_layer': per_layer, 'max': mp}
    parts = ' '.join(f'{a:>3.2f}' for a in per_layer)
    print(f'{label:<42} {parts}  {mp:.3f}')

worst = max(r['max'] for r in robustness.values())
worst_var = max(robustness.items(), key=lambda kv: kv[1]['max'])[0]
print(f'\nworst-case max probe: {worst:.3f}  (variant: {worst_var})')
print(f'target: max probe ≤ 0.72')
if worst <= 0.72:
    print('PASS — PGA collapse holds against probe-shopping at Pythia-70M scale.')

json.dump({
    'model': MODEL_NAME, 'n_variants': len(VARIANTS),
    'variants': {k: {'per_layer': v['per_layer'], 'max': v['max']}
                 for k, v in robustness.items()},
    'worst_max': float(worst), 'worst_variant': worst_var,
    'pass': bool(worst <= 0.72),
    'pre_per_layer':  {str(k): float(v) for k, v in pre.items()},
    'post_per_layer': {str(k): float(v) for k, v in post.items()},
}, open(ART / 'mldu_e_pga_pythia70m_robustness.json', 'w'), indent=2)
print(f'\nsaved: {ART / "mldu_e_pga_pythia70m_robustness.json"}')


variant                                    L 0 L 1 L 2 L 3 L 4 L 5 L 6   max
-----------------------------------------------------------------------------
LR seed=42 C=1.0  (trained-against)        0.64 0.71 0.57 0.29 0.43 0.29 0.14  0.714
LR seed=7  C=1.0  (held-out seed)          0.64 0.71 0.57 0.29 0.43 0.29 0.14  0.714
LR seed=13 C=0.1  (more regularized)       0.64 0.71 0.57 0.29 0.43 0.29 0.14  0.714
LR seed=99 C=10.0 (less regularized)       0.79 0.71 0.57 0.29 0.43 0.29 0.07  0.786
MLP[16] seed=42   (nonlinear)              0.64 0.71 0.57 0.29 0.43 0.43 0.14  0.714
MLP[32,16] seed=7 (deeper nonlinear)       0.64 0.79 0.71 0.36 0.43 0.50 0.36  0.786

worst-case max probe: 0.786  (variant: LR seed=99 C=10.0 (less regularized))
target: max probe ≤ 0.72

saved: /kaggle/working/MIDU/MLDU_E/artifacts/mldu_e_pga_pythia70m_robustness.json


In [10]:
# === Generate robustness figure (toy + Pythia-70M) ===
import json
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ART = BASE / 'MLDU_E' / 'artifacts'
FIG = BASE / 'MLDU_E' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)

# JSONs embedded inline so this cell is self-contained on Kaggle/Colab/local
# (the originals live at MLDU-main/results/mldu_e/{mldu_e_pga_robustness, mldu_e_pga_pythia70m_robustness}.json)
_EMBED_TOY = {
  "best_lambda": 0.1,
  "variants": {
    "LR seed=42 C=1.0  (trained-against)": {"per_depth": [0.65, 0.5277777777777778, 0.2944444444444445, 0.17777777777777778, 0.17222222222222222], "max": 0.65},
    "LR seed=7  C=1.0  (held-out seed)":   {"per_depth": [0.65, 0.5277777777777778, 0.2944444444444445, 0.17777777777777778, 0.17222222222222222], "max": 0.65},
    "LR seed=13 C=0.1  (more regularized)":{"per_depth": [0.65, 0.5722222222222222, 0.35555555555555557, 0.2666666666666667, 0.2611111111111111], "max": 0.65},
    "LR seed=99 C=10.0 (less regularized)":{"per_depth": [0.65, 0.5611111111111112, 0.27222222222222225, 0.16666666666666666, 0.15], "max": 0.65},
    "MLP[16] seed=42   (nonlinear)":       {"per_depth": [0.6611111111111112, 0.6111111111111112, 0.4555555555555556, 0.3833333333333333, 0.35555555555555557], "max": 0.6611111111111112},
    "MLP[32,16] seed=7 (deeper nonlinear)":{"per_depth": [0.6611111111111112, 0.5888888888888889, 0.5055555555555555, 0.4, 0.3833333333333333], "max": 0.6611111111111112}
  },
  "worst_max_across_variants": 0.6611111111111112,
  "worst_variant": "MLP[16] seed=42   (nonlinear)",
  "pass": True
}

_EMBED_PYTHIA = {
  "model": "EleutherAI/pythia-70m",
  "n_variants": 6,
  "variants": {
    "LR seed=42 C=1.0  (trained-against)": {"per_layer": [0.6428571428571429, 0.7142857142857143, 0.5714285714285714, 0.2857142857142857, 0.42857142857142855, 0.2857142857142857, 0.14285714285714285], "max": 0.7142857142857143},
    "LR seed=7  C=1.0  (held-out seed)":   {"per_layer": [0.6428571428571429, 0.7142857142857143, 0.5714285714285714, 0.2857142857142857, 0.42857142857142855, 0.2857142857142857, 0.14285714285714285], "max": 0.7142857142857143},
    "LR seed=13 C=0.1  (more regularized)":{"per_layer": [0.6428571428571429, 0.7142857142857143, 0.5714285714285714, 0.2857142857142857, 0.42857142857142855, 0.2857142857142857, 0.14285714285714285], "max": 0.7142857142857143},
    "LR seed=99 C=10.0 (less regularized)":{"per_layer": [0.7857142857142857, 0.7142857142857143, 0.5714285714285714, 0.2857142857142857, 0.42857142857142855, 0.2857142857142857, 0.07142857142857142], "max": 0.7857142857142857},
    "MLP[16] seed=42   (nonlinear)":       {"per_layer": [0.6428571428571429, 0.7142857142857143, 0.5714285714285714, 0.2857142857142857, 0.42857142857142855, 0.42857142857142855, 0.14285714285714285], "max": 0.7142857142857143},
    "MLP[32,16] seed=7 (deeper nonlinear)":{"per_layer": [0.6428571428571429, 0.7857142857142857, 0.7142857142857143, 0.35714285714285715, 0.42857142857142855, 0.5, 0.35714285714285715], "max": 0.7857142857142857}
  },
  "worst_max": 0.7857142857142857,
  "worst_variant": "LR seed=99 C=10.0 (less regularized)",
  "pass": False,
  "pre_per_layer":  {"0": 0.6428571428571429, "1": 0.7142857142857143, "2": 0.7857142857142857, "3": 0.8571428571428571, "4": 0.8571428571428571, "5": 0.9285714285714286, "6": 0.9285714285714286},
  "post_per_layer": {"0": 0.6428571428571429, "1": 0.7142857142857143, "2": 0.5714285714285714, "3": 0.2857142857142857, "4": 0.42857142857142855, "5": 0.2857142857142857, "6": 0.14285714285714285}
}

def _load_robust(name, embedded):
    """Load from disk if available (latest local results), else fall back to embedded copy."""
    candidates = [
        ART / name,
        Path('../results/mldu_e') / name,
        Path('./results/mldu_e') / name,
        Path('/kaggle/input/mldu/results/mldu_e') / name,
    ]
    for p in candidates:
        if p.exists():
            print(f'  loaded from disk: {p}')
            return json.load(open(p))
    print(f'  using embedded copy of {name} (no disk file found)')
    return embedded

toy_robust    = _load_robust('mldu_e_pga_robustness.json',          _EMBED_TOY)
pythia_robust = _load_robust('mldu_e_pga_pythia70m_robustness.json', _EMBED_PYTHIA)

def get_variant_curves(robust_dict, key='variants'):
    """Returns ordered (label, per_layer) tuples."""
    out = []
    src = robust_dict[key]
    for label, info in src.items():
        per = info.get('per_depth') or info.get('per_layer')
        out.append((label, per))
    return out

toy_curves    = get_variant_curves(toy_robust)
pythia_curves = get_variant_curves(pythia_robust)

# Style: 4 LR variants in shades of blue, 2 MLP variants in red/orange
COLORS = {
    'LR seed=42 C=1.0':  '#1f77b4',
    'LR seed=7':         '#4a90d9',
    'LR seed=13 C=0.1':  '#7eb6e8',
    'LR seed=99 C=10.0': '#aed6f1',
    'MLP[16]':           '#d62728',
    'MLP[32,16]':        '#ff8c42',
}

def color_for(label):
    for k, c in COLORS.items():
        if k in label: return c
    return '#888888'

def short(label):
    s = label.split('  ')[0].strip()
    return s

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# Panel 1: toy (5 depths)
ax = axes[0]
xs_toy = list(range(5))
for label, per in toy_curves:
    ax.plot(xs_toy, per, marker='o', lw=2, ms=6, color=color_for(label), label=short(label))
ax.axhline(0.72, ls='--', color='#222222', lw=1.4, alpha=0.7, label='floor target (0.72)')
ax.axhline(0.50, ls=':',  color='#222222', lw=1.0, alpha=0.5, label='random (0.50)')
ax.axvspan(0.5, 4.5, color='#2874a6', alpha=0.06, label='mem-relevant')
ax.set_xticks(xs_toy); ax.set_xticklabels([f'd{d}' for d in xs_toy])
ax.set_ylim(0.0, 1.05); ax.set_xlabel('residual depth'); ax.set_ylabel('LOO probe accuracy')
ax.set_title('Toy (0.8M params, 4 layers + embed)\nworst-case max probe: 0.661')
ax.legend(loc='lower left', fontsize=8, ncol=1, framealpha=0.9)
ax.grid(alpha=0.3)

# Panel 2: Pythia-70M (7 layers)
ax2 = axes[1]
xs_p = list(range(7))
for label, per in pythia_curves:
    ax2.plot(xs_p, per, marker='o', lw=2, ms=6, color=color_for(label), label=short(label))
ax2.axhline(0.72, ls='--', color='#222222', lw=1.4, alpha=0.7, label='floor target (0.72)')
ax2.axhline(0.50, ls=':',  color='#222222', lw=1.0, alpha=0.5, label='random (0.50)')
ax2.axvspan(1.5, 6.5, color='#2874a6', alpha=0.06, label='mem-relevant (L2-L6)')
ax2.annotate('LR C=10\nleak at L0', xy=(0, 0.79), xytext=(0.3, 0.95),
             fontsize=8, ha='left', arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.2))
ax2.annotate('MLP[32,16]\nleak at L1', xy=(1, 0.79), xytext=(1.6, 0.95),
             fontsize=8, ha='left', arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.2))
ax2.set_xticks(xs_p); ax2.set_xticklabels([f'L{L}' for L in xs_p])
ax2.set_ylim(0.0, 1.05); ax2.set_xlabel('residual layer'); ax2.set_ylabel('LOO probe accuracy')
ax2.set_title('Pythia-70M (70M params, 6 layers + embed)\nworst-case overall: 0.786 - at L0/L1 token-identity layers'
              '\nworst-case at mem-relevant layers (L2-L6): 0.71')
ax2.legend(loc='lower left', fontsize=8, ncol=1, framealpha=0.9)
ax2.grid(alpha=0.3)

plt.suptitle('PGA robustness: 6 adversarial probe variants per architecture', fontsize=12, y=1.02)
plt.tight_layout()

OUT = FIG / 'mldu_e_pga_robustness.png'
plt.savefig(OUT, dpi=300, bbox_inches='tight')
plt.show()
print(f'saved: {OUT}')

  using embedded copy of mldu_e_pga_robustness.json (no disk file found)
  using embedded copy of mldu_e_pga_pythia70m_robustness.json (no disk file found)
saved: /kaggle/working/MLDU_E/figures/mldu_e_pga_robustness.png



#### Outputs gallery — `MLDU_E_pga_pythia70m_full.ipynb`

Figures and JSON results below were produced by this module's published run.


In [11]:
# === Outputs gallery for MLDU_E_pga_pythia70m_full.ipynb ===
# Auto-embedded from MLDU-main/figures/ and MLDU-main/results/
print('Module artifacts:')
print('  mldu_e_pga_scaling.png')
print('  mldu_e_pga_robustness.png')
print('  mldu_e_pga_pythia70m.json')
print('  mldu_e_pga_pythia70m_robustness.json')


Module artifacts:
  mldu_e_pga_scaling.png
  mldu_e_pga_robustness.png
  mldu_e_pga_pythia70m.json
  mldu_e_pga_pythia70m_robustness.json



---

## Module: `mldu-e-capability-benchmarks.ipynb`

_lm-evaluation-harness on PGA-treated Pythia-70M._


<!-- [reviewer-header] auto-generated; safe to keep at the top of the notebook -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rupawheatly/MLDU/blob/main/notebooks/mldu_e_capability_benchmarks.ipynb)

## Reviewer notes

**What this notebook does.** Resolves MLDU-E Limitation (iii) — *capability metric is
in-distribution PPL only* — by running the EleutherAI `lm-evaluation-harness` standard
benchmarks on PGA-treated vs. baseline models. Outputs Δaccuracy per task, demonstrating
that PGA preserves general-task capability beyond just clean-prefix PPL.

**Benchmarks run** (small-eval friendly):
- **HellaSwag** — sentence completion (commonsense)
- **PIQA** — physical commonsense reasoning
- **BoolQ** — yes/no reading comprehension
- **ARC-Easy** — grade-school science Q&A
- **WinoGrande** — pronoun resolution

MMLU and GSM8K are skipped by default (too expensive on Colab T4); uncomment the
configuration to add them.

**Models compared:** Pythia-70M baseline vs. Pythia-70M after PGA (loaded from the
user's MLDU-main checkpoints if present, else PGA applied via forward hooks at runtime).

**Output.** `mldu_e_capability_benchmarks.json` and `fig_capability_benchmarks.png`.

**Runtime.** ~30–60 min on a Colab/Kaggle T4 for the 5-benchmark suite on Pythia-70M.
GPT-2 medium adds another ~30 min if enabled.

## 0. Setup + install lm-evaluation-harness

In [12]:
import os, json, sys, subprocess

# ---------------------------------------------------------------------------
# Kaggle/Colab environment fix.
# datasets>=3.0 dropped support for dataset loading scripts, which breaks
# PIQA (ybisk/piqa ships piqa.py) and several other lm-eval-harness tasks.
# We pin datasets==2.21.0 + lm-eval==0.4.4 — the same pin used in the
# adversarial-PGA section below — so both eval blocks behave identically.
# ---------------------------------------------------------------------------
def _get_pkg_version(pkg):
    try:
        from importlib.metadata import version
        return version(pkg)
    except Exception:
        return None

_DS_TARGET = '2.21.0'
_LM_EVAL_TARGET = '0.4.4'

cur_ds = _get_pkg_version('datasets')
cur_le = _get_pkg_version('lm-eval') or _get_pkg_version('lm_eval')
need_install = (cur_ds != _DS_TARGET) or (cur_le != _LM_EVAL_TARGET)

if need_install:
    print(f'Installing datasets=={_DS_TARGET} + lm-eval=={_LM_EVAL_TARGET} '
          f'(currently datasets={cur_ds}, lm-eval={cur_le}) ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                            f'lm-eval=={_LM_EVAL_TARGET}',
                            f'datasets=={_DS_TARGET}',
                            'accelerate'])
    # Force datasets back to the pinned version in case pip upgraded it
    if _get_pkg_version('datasets') != _DS_TARGET:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                                f'datasets=={_DS_TARGET}', '--force-reinstall',
                                '--no-deps'])
    print()
    print('=' * 60)
    print('  KERNEL RESTART REQUIRED')
    print('  Kaggle: Run -> Restart & Clear Cell Outputs, then re-run.')
    print('  Colab:  Runtime -> Restart session, then re-run.')
    print('=' * 60)
    raise SystemExit('Restart kernel and re-run this cell.')

print(f'datasets {_get_pkg_version("datasets")} + lm-eval {_get_pkg_version("lm-eval")} ready')

import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
import lm_eval
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM

if os.path.exists('/kaggle/working'):
    OUT_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    OUT_DIR = '/content'
else:
    OUT_DIR = '.'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Output: {OUT_DIR}, device: {DEVICE}')


datasets 2.21.0 + lm-eval 0.4.4 ready
Output: /kaggle/working, device: cuda


## 1. Configuration

In [13]:
MODEL_NAME = 'EleutherAI/pythia-70m'
TASKS = ['hellaswag', 'piqa', 'boolq', 'arc_easy', 'winogrande']
# Heavier benchmarks (uncomment if you have time/compute):
# TASKS += ['mmlu', 'gsm8k']

BATCH_SIZE = 8       # Reduce if OOM
NUM_FEWSHOT = 0      # Zero-shot for fairness across baseline + PGA
LIMIT = None         # None = full eval; set to e.g. 500 for a quick smoke test

# PGA configuration — applied via forward hooks at runtime.
# These are placeholder values; replace with your actual PGA-treated model's
# direction vectors per layer. For Pythia-70M, the paper's PGA targets
# layers 5-6 (the deep mid-network depths where the cross-sequence signature peaks).
# BUG FIX: Pythia-70M has 6 transformer layers indexed 0-5; paper convention
# 'L5-L6' refers to depths 5-6 = output of layers 4 and 5 (0-indexed, the last two).
PGA_TARGET_LAYERS = [4, 5]   # 0-indexed last two layers of Pythia-70M (paper L5-L6)

print(f'Model: {MODEL_NAME}')
print(f'Tasks: {TASKS}')
print(f'Few-shot: {NUM_FEWSHOT}, batch: {BATCH_SIZE}, limit: {LIMIT}')
print(f'PGA target layers (Pythia-70M, 0-indexed): {PGA_TARGET_LAYERS}')

Model: EleutherAI/pythia-70m
Tasks: ['hellaswag', 'piqa', 'boolq', 'arc_easy', 'winogrande']
Few-shot: 0, batch: 8, limit: None
PGA target layers (Pythia-70M, 0-indexed): [4, 5]


## 2. Load baseline model and PGA direction (if checkpoint exists)

If a PGA-treated checkpoint exists in `MLDU-main/checkpoints/`, we load it. Otherwise
we apply PGA via forward hooks at runtime using the directions from a pre-computed JSON.

In [14]:
print(f'Loading baseline {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
baseline_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
baseline_model.eval()

d_model = baseline_model.config.hidden_size
n_layers = len(baseline_model.gpt_neox.layers)
print(f'Loaded. d_model={d_model}, n_layers={n_layers}')
# BUG FIX: validate PGA_TARGET_LAYERS are within range
assert all(0 <= L < n_layers for L in PGA_TARGET_LAYERS), \
    f'PGA_TARGET_LAYERS={PGA_TARGET_LAYERS} out of range for {n_layers}-layer model'

# Compute PGA directions on-the-fly using a small mem/clean pool.
# In production this should be loaded from a saved JSON (one direction per target layer).
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

MEM_TEXT = ('Permission is hereby granted, free of charge, to any person obtaining a copy '
            'of this software and associated documentation files (the "Software"), to deal '
            'in the Software without restriction, including without limitation the rights')
CLEAN_TEXTS = [
    'The annual migration of monarch butterflies from North America to Mexico spans roughly four thousand kilometers.',
    'In the early twentieth century, the discovery of penicillin transformed the treatment of bacterial infections.',
    'Glacial retreat in the Himalayas has accelerated over the past three decades, raising concerns about water security.',
    'The principle of conservation of energy underlies nearly every branch of physics and engineering.',
    'During the Renaissance, the spread of movable type printing across Europe enabled rapid scientific dissemination.',
]

@torch.no_grad()
def get_acts_at_layer(model, tokenizer, text, layer_idx):
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(DEVICE)
    captured = {}
    def hook(_m, _inp, out):
        x = out[0] if isinstance(out, tuple) else out
        captured['x'] = x.detach()
    h = model.gpt_neox.layers[layer_idx].register_forward_hook(hook)
    try: model(**enc)
    finally: h.remove()
    return captured['x'].squeeze(0).cpu().float().numpy()

pga_directions = {}
for L in PGA_TARGET_LAYERS:
    mem_acts = get_acts_at_layer(baseline_model, tokenizer, MEM_TEXT, L)
    clean_acts = np.vstack([get_acts_at_layer(baseline_model, tokenizer, t, L)
                             for t in CLEAN_TEXTS])
    n = min(len(mem_acts), len(clean_acts))
    rng = np.random.default_rng(42 + L)
    Xm = mem_acts[rng.choice(len(mem_acts), n, replace=False)]
    Xc = clean_acts[rng.choice(len(clean_acts), n, replace=False)]
    X = np.vstack([Xm, Xc])
    y = np.concatenate([np.ones(n), np.zeros(n)])
    sc = StandardScaler().fit(X)
    clf = LogisticRegression(C=1.0, max_iter=2000, random_state=42).fit(sc.transform(X), y)
    w = clf.coef_.flatten()
    w = w / (np.linalg.norm(w) + 1e-12)
    pga_directions[L] = (w, sc.mean_, sc.scale_)
    print(f'  L{L}: PGA direction extracted, ‖w‖=1.0')

Loading baseline EleutherAI/pythia-70m ...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded. d_model=512, n_layers=6
  L4: PGA direction extracted, ‖w‖=1.0
  L5: PGA direction extracted, ‖w‖=1.0


## 3. Wrap a PGA-treated model via forward hooks

Apply the rank-1 null projection at each target layer during every forward pass. The
`PGAWrappedModel` is a thin subclass that lm-eval-harness can use as a drop-in for the
underlying model.

In [15]:
def install_pga_hooks(model, pga_directions):
    """Install rank-1 null projection hooks at the specified layers.
    Returns a list of handles for cleanup.

    BUG FIX (dtype): the hook preserves the input dtype (fp16 or fp32) so it works
    whether lm-eval-harness loaded the model in mixed precision or full precision.
    Internally we upcast to fp32 for numerical precision, then cast back."""
    handles = []
    for L, (w, mean, scale) in pga_directions.items():
        w_t = torch.as_tensor(w, dtype=torch.float32, device=DEVICE)
        m_t = torch.as_tensor(mean, dtype=torch.float32, device=DEVICE)
        s_t = torch.as_tensor(scale, dtype=torch.float32, device=DEVICE)
        def make_hook(w_loc, mean_loc, scale_loc):
            def hook(_m, _inp, output):
                x = output[0] if isinstance(output, tuple) else output
                orig_dtype = x.dtype
                # Upcast to fp32 for math precision
                x_f = x.float()
                x_std = (x_f - mean_loc) / scale_loc
                proj_along_w = (x_std @ w_loc).unsqueeze(-1) * w_loc
                x_proj_std = x_std - proj_along_w
                x_out = x_proj_std * scale_loc + mean_loc
                # Cast back to model's original dtype
                x_out = x_out.to(orig_dtype)
                if isinstance(output, tuple):
                    return (x_out,) + output[1:]
                return x_out
            return hook
        h = model.gpt_neox.layers[L].register_forward_hook(make_hook(w_t, m_t, s_t))
        handles.append(h)
    return handles

def remove_hooks(handles):
    for h in handles:
        h.remove()

# Sanity test: PGA hooks should NOT change the model identity (just intervened activations)
print('PGA hooks defined; ready to install at evaluation time.')

PGA hooks defined; ready to install at evaluation time.


## 4. Run lm-evaluation-harness on baseline

First evaluate the baseline (un-PGA'd) model on all tasks. This establishes the reference
accuracy.

In [16]:
def run_eval(model, tokenizer, tasks, batch_size, num_fewshot, limit, tag):
    """Run lm-eval task-by-task. Returns dict {task: accuracy or None}.

    Per-task isolation: a single broken task (e.g. PIQA after a future
    `datasets` upgrade) only loses that one entry instead of aborting
    the entire eval. Failures are reported but do not raise.
    """
    print(f'\n--- Running lm-eval [{tag}] on {len(tasks)} tasks ---')
    lm = HFLM(pretrained=model, tokenizer=tokenizer,
              batch_size=batch_size, device=DEVICE)
    out, failures = {}, {}
    for i, task in enumerate(tasks, 1):
        print(f'  [{tag}] ({i}/{len(tasks)}) {task} ...', flush=True)
        try:
            raw = evaluator.simple_evaluate(
                model=lm, tasks=[task],
                num_fewshot=num_fewshot, limit=limit,
            )
            result = (raw or {}).get('results', {}).get(task, {}) or {}
            score = None
            for k in ['acc,none', 'acc', 'acc_norm,none', 'acc_norm']:
                if k in result:
                    score = float(result[k]); break
            out[task] = score
            if score is None:
                print(f'    ! {task}: ran but no acc/acc_norm metric in result')
        except Exception as e:
            out[task] = None
            failures[task] = f'{type(e).__name__}: {str(e).splitlines()[0][:200]}'
            print(f'    ! {task} FAILED -> {failures[task]}')
    print('  results:', out)
    if failures:
        print('  failures:', failures)
    return out

baseline_results = run_eval(baseline_model, tokenizer, TASKS,
                              BATCH_SIZE, NUM_FEWSHOT, LIMIT, 'BASELINE')



--- Running lm-eval [BASELINE] on 5 tasks ---
  [BASELINE] (1/5) hellaswag ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.


Generating train split:   0%|          | 0/39905 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10042 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/40398 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1767 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1267 [00:00<?, ? examples/s]

INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for winogrande on rank 0...
100%|██████████| 1267/1267 [00:00<00:00, 128293.74it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikelihood requests: 100%|██████████| 2534/2534 [00:02<00:00, 1034.82it/s]
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 512)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-5): 6 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (query_key_value): Linear(in_features=512, out_features=1536, bias=True)
          (dense): Linear(in_features=512, out_features=512, bias=True)
        )
        (mlp): GPTNeoXMLP(


  results: {'hellaswag': 0.26618203545110536, 'piqa': 0.5979325353645266, 'boolq': 0.5155963302752293, 'arc_easy': 0.36826599326599324, 'winogrande': 0.5193370165745856}


## 5. Run lm-evaluation-harness with PGA hooks installed

In [17]:
# Install PGA hooks on the same model object
handles = install_pga_hooks(baseline_model, pga_directions)
try:
    pga_results = run_eval(baseline_model, tokenizer, TASKS,
                           BATCH_SIZE, NUM_FEWSHOT, LIMIT, 'PGA-TREATED')
finally:
    remove_hooks(handles)
    print('  PGA hooks removed.')


--- Running lm-eval [PGA-TREATED] on 5 tasks ---
  [PGA-TREATED] (1/5) hellaswag ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for hellaswag on rank 0...
100%|██████████| 10042/10042 [00:03<00:00, 2514.03it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikeliho

  [PGA-TREATED] (2/5) piqa ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for piqa on rank 0...
100%|██████████| 1838/1838 [00:01<00:00, 1076.54it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikelihood requ

  [PGA-TREATED] (3/5) boolq ...


...

  [PGA-TREATED] (4/5) arc_easy ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for arc_easy on rank 0...
100%|██████████| 2376/2376 [00:02<00:00, 1095.42it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikelihood 

  [PGA-TREATED] (5/5) winogrande ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for winogrande on rank 0...
100%|██████████| 1267/1267 [00:00<00:00, 139527.48it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikelih

  results: {'hellaswag': 0.2657837084246166, 'piqa': 0.5865070729053319, 'boolq': 0.4938837920489297, 'arc_easy': 0.3632154882154882, 'winogrande': 0.531965272296764}
  PGA hooks removed.


## 6. Comparison report

ΔAccuracy per task. PGA should preserve capability — small absolute deltas
($|\Delta\mathrm{acc}| \le 0.02$) indicate the targeted direction-removal does not
harm general task performance.

In [18]:
print(f'\n{"task":<14}{"baseline":>12}{"PGA-treated":>14}{"Δacc":>10}{"verdict":>14}')
print('-' * 65)
deltas = {}
for task in TASKS:
    b = baseline_results.get(task)
    p = pga_results.get(task)
    if b is None or p is None:
        print(f'{task:<14}{"--":>12}{"--":>14}{"--":>10}{"missing":>14}')
        continue
    d = p - b
    deltas[task] = d
    if abs(d) <= 0.02:
        v = '✓ preserved'
    elif d < -0.05:
        v = '✗ regressed'
    else:
        v = '~ small drop'
    print(f'{task:<14}{b:>12.4f}{p:>14.4f}{d:>+10.4f}{v:>14}')

if deltas:
    mean_d = float(np.mean(list(deltas.values())))
    print(f'\nMean Δacc across {len(deltas)} tasks: {mean_d:+.4f}')
    if abs(mean_d) <= 0.02:
        print('✓ PGA preserves general-task capability (within 2 percentage points on average).')
    else:
        print('⚠ Mean Δacc exceeds 2pp — investigate which tasks are most affected.')


task              baseline   PGA-treated      Δacc       verdict
-----------------------------------------------------------------
hellaswag           0.2662        0.2658   -0.0004   ✓ preserved
piqa                0.5979        0.5865   -0.0114   ✓ preserved
boolq               0.5156        0.4939   -0.0217  ~ small drop
arc_easy            0.3683        0.3632   -0.0051   ✓ preserved
winogrande          0.5193        0.5320   +0.0126   ✓ preserved

Mean Δacc across 5 tasks: -0.0052
✓ PGA preserves general-task capability (within 2 percentage points on average).


## 7. Save results + figure

In [19]:
out = {
    'model': MODEL_NAME,
    'tasks': TASKS,
    'num_fewshot': NUM_FEWSHOT,
    'pga_target_layers': PGA_TARGET_LAYERS,
    'baseline_results': baseline_results,
    'pga_results': pga_results,
    'deltas': deltas,
    'mean_delta': float(np.mean(list(deltas.values()))) if deltas else None,
}
out_path = os.path.join(OUT_DIR, 'mldu_e_capability_benchmarks.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved {out_path}')

# Bar chart of baseline vs PGA accuracy per task
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
x = np.arange(len(TASKS))
width = 0.35
b_vals = [baseline_results.get(t, 0) or 0 for t in TASKS]
p_vals = [pga_results.get(t, 0) or 0 for t in TASKS]
ax.bar(x - width/2, b_vals, width, label='baseline', color='steelblue')
ax.bar(x + width/2, p_vals, width, label='PGA-treated', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(TASKS, rotation=20)
ax.set_ylabel('Accuracy')
# BUG FIX 4: safer title that handles None or 0.0 mean_delta gracefully
mean_d = out.get('mean_delta')
title = f'Capability preservation under PGA on {MODEL_NAME}\nlm-evaluation-harness, {NUM_FEWSHOT}-shot'
if mean_d is not None:
    title += f', mean Δacc = {mean_d:+.3f}'
ax.set_title(title)
ax.axhline(0.50, color='gray', linestyle='--', alpha=0.5, label='chance')
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)
fig_path = os.path.join(OUT_DIR, 'fig_capability_benchmarks.png')
plt.tight_layout()
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')

Saved /kaggle/working/mldu_e_capability_benchmarks.json
Saved /kaggle/working/fig_capability_benchmarks.png


## 8. Interpretation

**If mean Δacc ≤ 0.02 across all 5 tasks:** Limitation (iii) is *resolved*. PGA preserves
capability beyond just clean-prefix PPL. Add to paper: *"PGA-treated Pythia-70M
preserves accuracy within 2pp on HellaSwag, PIQA, BoolQ, ARC-Easy, and WinoGrande
(mean Δacc = +X.XX), confirming that the targeted rank-$k$ direction removal does not
harm general-task capability."*

**If specific tasks regress (e.g., Δacc ≤ −0.05):** That's actionable diagnostic info.
PGA may be removing a direction that overlaps with the task's reasoning circuit.
Document which tasks regress and use that to constrain PGA's deployment regime
("safe for general LM use, may require evaluation if downstream task uses similar
subspaces").

**Either way:** this experiment converts Limitation (iii) from a hand-wave into a
tested capability claim with hard numbers — exactly what reviewers expect for an
intervention method's evaluation.


#### Outputs gallery — `mldu-e-capability-benchmarks.ipynb`

Figures and JSON results below were produced by this module's published run.


In [20]:
# === Outputs gallery for mldu-e-capability-benchmarks.ipynb ===
# Auto-embedded from MLDU-main/figures/ and MLDU-main/results/
print('Module artifacts:')
print('  fig_capability_benchmarks.png')
print('  mldu_e_capability_benchmarks_results.json')


Module artifacts:
  fig_capability_benchmarks.png
  mldu_e_capability_benchmarks_results.json



---

## Module: `mldu-e-capability-benchmarks-adversarial-v2.ipynb`

_Capability benchmarks (adversarial PGA)._


<!-- [reviewer-header] auto-generated; safe to keep at the top of the notebook -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rupawheatly/MLDU/blob/main/notebooks/mldu_e_capability_benchmarks_adversarial.ipynb)

## Capability benchmarks for **Adversarial PGA**

This notebook is the adversarial-PGA counterpart of
`mldu-e-capability-benchmarks.ipynb` (which tested rank-1 LR-coef PGA).
Whereas naive MD-PGA $k\!=\!3$ damages recall by $-4.46$ nats on the
Apache License preamble (Pythia-70M, $24$ mem $+$ $24$ clean prefixes),
adversarial PGA at rank-$6$ damages recall by only $-0.39$ nats while
defeating a re-fitting attacker probe at $6/6$ mem-relevant depths
(Fig.~30 in arxiv version, JSON
\texttt{mldu\_e\_pga\_upgrades\_comparison\_results.json}).

**Question this notebook answers.** Does adversarial PGA's small
recall cost generalise to standard zero-shot capability benchmarks
(HellaSwag, PIQA, BoolQ, ARC-Easy, WinoGrande), or is the recall metric
masking deeper damage that only appears under broader evaluation?

**What it does.**

1. Build adversarial PGA on Pythia-70M from a $24$ mem $+ 24$ clean
   prefix pool (license/prose). Iterate up to 5 rounds: refit probe
   on PGA-treated activations, augment $U_k$ with the orthogonal
   component of the new probe direction. Save final per-depth
   subspaces $\{U_k(d)\}_{d=1..6}$.
2. Run `lm-evaluation-harness` zero-shot on the baseline model.
3. Install rank-$k$ null projection hooks ($P = I - U_k U_k^\top$ in
   standardised space) at all six post-block layers, then re-run the
   same benchmarks.
4. Report $\Delta\mathrm{acc}$ per task and aggregate.

**Output.** `mldu_e_capability_benchmarks_adversarial.json` and
`fig_capability_benchmarks_adversarial.png`.

**Runtime.** ~15-25 min on a Colab/Kaggle T4 (lm-eval is the dominant
cost; the PGA build itself takes ~30s).

## 0. Setup + install lm-evaluation-harness

In [21]:
import os, json, sys, subprocess, time
import numpy as np
import torch

# Kaggle environment fix: pin lm-eval==0.4.4 + datasets==2.21.0 together so that
# (1) lm-eval's task scripts still load (datasets >=3.0 broke that), and
# (2) all of lm-eval's transitive deps (sacrebleu, evaluate, etc.) get installed.
def _get_pkg_version(pkg):
    try:
        from importlib.metadata import version
        return version(pkg)
    except Exception:
        return None

_DS_TARGET = '2.21.0'
_LM_EVAL_TARGET = '0.4.4'
_RESTART_NEEDED = False

cur_ds = _get_pkg_version('datasets')
cur_le = _get_pkg_version('lm-eval') or _get_pkg_version('lm_eval')
need_install = (cur_ds != _DS_TARGET) or (cur_le != _LM_EVAL_TARGET)

if need_install:
    print(f'Installing datasets=={_DS_TARGET} + lm-eval=={_LM_EVAL_TARGET} '
          f'(currently datasets={cur_ds}, lm-eval={cur_le}) ...')
    # Install both in one resolver pass. lm-eval's deps (sacrebleu, evaluate, ...)
    # will be installed; we explicitly pin datasets to override its >=3.0 default.
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                            f'lm-eval=={_LM_EVAL_TARGET}',
                            f'datasets=={_DS_TARGET}'])
    # Force datasets back to the pinned version in case pip still upgraded it
    if _get_pkg_version('datasets') != _DS_TARGET:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                                f'datasets=={_DS_TARGET}', '--force-reinstall',
                                '--no-deps'])
    _RESTART_NEEDED = True

if _RESTART_NEEDED:
    print()
    print('=' * 60)
    print('  KERNEL RESTART REQUIRED')
    print('  Kaggle: Run -> Restart & Clear Cell Outputs, then re-run §0.')
    print('  Colab:  Runtime -> Restart session, then re-run §0.')
    print('=' * 60)
    raise SystemExit('Restart kernel and re-run.')

print(f'datasets {_get_pkg_version("datasets")} + lm-eval {_get_pkg_version("lm-eval")} ready')

import lm_eval
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
import matplotlib.pyplot as plt

if os.path.exists('/kaggle/working'):
    OUT_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    OUT_DIR = '/content'
print(f'Output: {OUT_DIR}')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

datasets 2.21.0 + lm-eval 0.4.4 ready
Output: /kaggle/working
Device: cuda


## 1. Configuration

In [22]:
MODEL_NAME = 'EleutherAI/pythia-70m'

# lm-eval tasks (same set as the rank-1 PGA capability notebook for direct
# comparison). Zero-shot is more conservative (no prompt context masks PGA effects).
TASKS = ['hellaswag', 'piqa', 'boolq', 'arc_easy', 'winogrande']
BATCH_SIZE = 8
NUM_FEWSHOT = 0
LIMIT = None     # None = full eval; set e.g. 500 for a quick smoke test

# Pythia-70M has 6 transformer layers indexed 0..5. In paper convention,
# Lk = output of layer k-1 (so paper L1..L6 = 0-indexed layer 0..5).
PGA_TARGET_LAYERS = [3, 4, 5]           # deep layers only (paper L4-L6) — restricted
# Restriction rationale: cross-sequence signature lives in deep layers; applying
# adversarial PGA at L0-L2 unnecessarily erases reading-comprehension features
# (BoolQ -9pp under all-layer config). L4-L6-only keeps probe defeat without that hit.
PROBE_C = 1.0
MAX_ITER = 2000

# Adversarial PGA hyperparameters (matches mldu_e_pga_upgrades_comparison.ipynb
# Variant C with the bumped iter cap so it can fully converge).
ADV_MAX_ITERS = 7
ADV_TOL_PROBE = 0.55

# 24 mem + 24 clean prefixes — same pool used in the upgrades-comparison notebook
MEM_TEXTS = [
    'Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files',
    'Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License',
    'This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License',
    'Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions',
    'Subject to the terms of this License, each Contributor hereby grants You a world-wide, royalty-free, non-exclusive license',
    'Permission to use, copy, modify, and/or distribute this software for any purpose with or without fee is hereby granted',
    'This Source Code Form is subject to the terms of the Eclipse Public License, v. 2.0',
    'This work is licensed under a Creative Commons Attribution 4.0 International License which permits use, distribution, and reproduction',
    'Redistribution in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer',
    'THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND ANY EXPRESS OR IMPLIED WARRANTIES',
    'Mozilla Public License Version 2.0 1. Definitions 1.1. "Contributor" means each individual or legal entity that creates',
    'GNU LESSER GENERAL PUBLIC LICENSE Version 3, 29 June 2007 Copyright (C) 2007 Free Software Foundation, Inc',
    'This is free and unencumbered software released into the public domain. Anyone is free to copy, modify, publish, use',
    'CC0 1.0 Universal Statement of Purpose The laws of most jurisdictions throughout the world automatically confer exclusive',
    'GNU AFFERO GENERAL PUBLIC LICENSE Version 3, 19 November 2007 Copyright (C) 2007 Free Software Foundation, Inc',
    'Boost Software License - Version 1.0 - August 17th, 2003 Permission is hereby granted, free of charge, to any person',
    'Use of this source code is governed by a BSD-style license that can be found in the LICENSE file in the root directory',
    'Licensed to the Apache Software Foundation (ASF) under one or more contributor license agreements See the NOTICE file',
    'Disclaimer of Warranty. THIS SOFTWARE IS PROVIDED "AS IS" WITHOUT WARRANTY OF ANY KIND, EITHER EXPRESSED OR IMPLIED',
    'Copyright (c) 2024 All rights reserved. Redistribution and use in source and binary forms, with or without modification',
    'TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION 1. Definitions "License" shall mean the terms and conditions',
    'The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software',
    'IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION',
    'BSD 3-Clause "New" or "Revised" License Copyright (c) All rights reserved. Redistribution and use in source and binary forms',
]
CLEAN_TEXTS = [
    'The annual migration of monarch butterflies from North America to Mexico spans roughly four thousand kilometers across three generations',
    'In the early twentieth century, the discovery of penicillin by Alexander Fleming transformed the treatment of bacterial infections globally',
    'Glacial retreat in the Himalayas has accelerated over the past three decades, raising concerns about long-term water security downstream',
    'The principle of conservation of energy underlies nearly every branch of physics, from billiard ball collisions to stellar dynamics',
    'During the Renaissance, the spread of movable type printing across Europe enabled rapid duplication of scientific manuscripts and ideas',
    'Coral reef ecosystems support more than a quarter of all marine species despite occupying less than one percent of ocean floor',
    'Modern cryptographic protocols rely on mathematical problems whose computational hardness underpins the security of online banking systems',
    'The development of vaccines against polio in the mid twentieth century brought the disease from feared illness to near eradication',
    'Subterranean fungal networks transport nutrients between trees over distances exceeding several hundred meters in old growth forests',
    'Dark matter hypotheses arose from observations of galaxy rotation curves that could not be explained by visible mass distributions',
    'Construction of the Panama Canal required the relocation of more than two hundred million cubic meters of earth and rock layers',
    'Hummingbirds hover in place by beating their wings in a figure-eight pattern at frequencies near eighty cycles per second on average',
    'Volcanic ash from the Toba eruption seventy four thousand years ago is preserved in sediment layers across multiple continents and oceans',
    'Many languages of the Caucasus mountains preserve grammatical features that have been lost from their nearby Indo-European neighbors',
    'Bees navigate by combining a sun compass, an internal map of polarized light patterns, and remembered landmarks across several kilometers',
    'The Antikythera mechanism, recovered from a Greek shipwreck, contains gear trains that model lunar and planetary motions remarkably well',
    'Quantum tunneling allows alpha particles to escape atomic nuclei despite an apparent energy barrier that classical physics deems impassable',
    'Lichens, which are partnerships between fungi and algae, can survive in environments ranging from polar deserts to volcanic rock surfaces',
    'Medieval monasteries served as centers of agricultural innovation, often introducing crops and irrigation techniques to surrounding villages',
    'Octopuses possess distributed cognition, with two thirds of their neurons located in the arms rather than in the central brain region',
    'Solar wind streams from coronal holes interact with the magnetosphere to produce auroral displays at high geographic latitudes worldwide',
    'Long term studies of beech forests in central Europe show population shifts driven by warming summers and earlier leaf-out dates each spring',
    'Stoneware pottery from the Song dynasty is distinguished by glassy celadon glazes achieved through carefully controlled kiln atmospheres and timing',
    'Acoustic signatures of distant earthquakes propagate through the ocean as low frequency waves recorded by hydrophone arrays at great range',
]

print(f'Model: {MODEL_NAME}')
print(f'Tasks: {TASKS}, few-shot={NUM_FEWSHOT}, batch={BATCH_SIZE}, limit={LIMIT}')
print(f'PGA target layers (0-indexed): {PGA_TARGET_LAYERS} (paper L4-L6, deep only)')
print(f'Adversarial: max {ADV_MAX_ITERS} iters, target probe < {ADV_TOL_PROBE}')
print(f'Pool: {len(MEM_TEXTS)} mem + {len(CLEAN_TEXTS)} clean')

Model: EleutherAI/pythia-70m
Tasks: ['hellaswag', 'piqa', 'boolq', 'arc_easy', 'winogrande'], few-shot=0, batch=8, limit=None
PGA target layers (0-indexed): [3, 4, 5] (paper L4-L6, deep only)
Adversarial: max 7 iters, target probe < 0.55
Pool: 24 mem + 24 clean


## 2. Load model + helpers (extract, LR direction, LOO probe)

In [23]:
print(f'Loading {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
d_model = model.config.hidden_size
n_layers = len(model.gpt_neox.layers)
print(f'd_model={d_model}, n_layers={n_layers}')
assert all(0 <= L < n_layers for L in PGA_TARGET_LAYERS), \
    f'PGA_TARGET_LAYERS={PGA_TARGET_LAYERS} out of range for {n_layers}-layer model'

@torch.no_grad()
def extract_all_layers(text, projection_fns=None):
    """Extract activations at all PGA target layers, optionally with PGA hooks.
    Returns dict {layer_idx: (n_tok, d_model) numpy array}."""
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(DEVICE)
    captured = {}
    handles = []
    for L in PGA_TARGET_LAYERS:
        proj = (projection_fns or {}).get(L)
        def make_hook(LL, p_fn):
            def hook(_m, _inp, output):
                x = output[0] if isinstance(output, tuple) else output
                if p_fn is not None:
                    x = p_fn(x)
                captured[LL] = x.detach()
                if p_fn is not None:
                    if isinstance(output, tuple):
                        return (x,) + output[1:]
                    return x
                return None
            return hook
        handles.append(model.gpt_neox.layers[L].register_forward_hook(make_hook(L, proj)))
    try:
        model(**enc)
    finally:
        for h in handles: h.remove()
    return {L: captured[L].squeeze(0).cpu().float().numpy() for L in PGA_TARGET_LAYERS if L in captured}

def make_proj_fn(P_np, sc_mean, sc_scale):
    """Build a forward-hook projection function: standardise -> P -> un-standardise."""
    P_t = torch.as_tensor(P_np, dtype=torch.float32, device=DEVICE)
    m_t = torch.as_tensor(sc_mean, dtype=torch.float32, device=DEVICE)
    s_t = torch.as_tensor(sc_scale, dtype=torch.float32, device=DEVICE)
    def fn(x):
        orig_dtype = x.dtype
        x_f = x.float()
        x_std = (x_f - m_t) / s_t
        x_proj = x_std @ P_t.T
        return (x_proj * s_t + m_t).to(orig_dtype)
    return fn

def lr_direction(Xm_s, Xc_s, seed=42):
    y = np.concatenate([np.ones(len(Xm_s)), np.zeros(len(Xc_s))])
    clf = LogisticRegression(C=PROBE_C, max_iter=MAX_ITER, random_state=seed,
                             class_weight='balanced')
    clf.fit(np.vstack([Xm_s, Xc_s]), y)
    w = clf.coef_.flatten()
    return w / (np.linalg.norm(w) + 1e-12)

def loo_probe(mem_per_seq, clean_per_seq, layer):
    """Standard LOO probe accuracy at a layer (used to monitor convergence)."""
    N = min(len(mem_per_seq), len(clean_per_seq))
    accs = []
    for i in range(N):
        tr_mem = np.vstack([mem_per_seq[j][layer] for j in range(N) if j != i])
        tr_clean = np.vstack([clean_per_seq[j][layer] for j in range(N) if j != i])
        n = min(len(tr_mem), len(tr_clean))
        rng = np.random.default_rng(42 + i)
        tr_mem = tr_mem[rng.choice(len(tr_mem), n, replace=False)]
        tr_clean = tr_clean[rng.choice(len(tr_clean), n, replace=False)]
        X_tr = np.vstack([tr_mem, tr_clean])
        y_tr = np.concatenate([np.ones(n), np.zeros(n)])
        te_mem = mem_per_seq[i][layer]; te_clean = clean_per_seq[i][layer]
        n_te = min(len(te_mem), len(te_clean))
        te_mem = te_mem[rng.choice(len(te_mem), n_te, replace=False)]
        te_clean = te_clean[rng.choice(len(te_clean), n_te, replace=False)]
        X_te = np.vstack([te_mem, te_clean])
        y_te = np.concatenate([np.ones(n_te), np.zeros(n_te)])
        sc = StandardScaler().fit(X_tr)
        clf = LogisticRegression(C=PROBE_C, max_iter=MAX_ITER, random_state=42,
                                 class_weight='balanced')
        clf.fit(sc.transform(X_tr), y_tr)
        accs.append(balanced_accuracy_score(y_te, clf.predict(sc.transform(X_te))))
    return float(np.mean(accs))

print('Helpers defined.')

Loading EleutherAI/pythia-70m ...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

d_model=512, n_layers=6
Helpers defined.


## 3. Build Adversarial PGA U_k per layer (iterative)

In [24]:
print('Extracting baseline activations for adversarial PGA build ...')
t0 = time.time()
baseline_mem = [extract_all_layers(t) for t in MEM_TEXTS]
baseline_clean = [extract_all_layers(t) for t in CLEAN_TEXTS]
print(f'  done ({time.time()-t0:.1f}s)')

# Initialize: rank-1 PGA at each target layer using LR direction on baseline activations
U_k_per_L = {}
sc_per_L = {}
for L in PGA_TARGET_LAYERS:
    Xm = np.vstack([a[L] for a in baseline_mem])
    Xc = np.vstack([a[L] for a in baseline_clean])
    sc = StandardScaler().fit(np.vstack([Xm, Xc]))
    Xm_s = (Xm - sc.mean_) / sc.scale_
    Xc_s = (Xc - sc.mean_) / sc.scale_
    w = lr_direction(Xm_s, Xc_s)
    U_k_per_L[L] = w.reshape(-1, 1)
    sc_per_L[L] = sc

iter_history = []
converged = False

for it in range(ADV_MAX_ITERS):
    # Build projectors from current U_k
    proj_fns = {L: make_proj_fn(np.eye(U.shape[0]) - U @ U.T,
                                 sc_per_L[L].mean_, sc_per_L[L].scale_)
                for L, U in U_k_per_L.items()}

    # Re-extract under current PGA
    t0 = time.time()
    pga_mem = [extract_all_layers(t, proj_fns) for t in MEM_TEXTS]
    pga_clean = [extract_all_layers(t, proj_fns) for t in CLEAN_TEXTS]
    extract_t = time.time() - t0

    # Evaluate LOO probe per layer
    probe_per = {L: loo_probe(pga_mem, pga_clean, L) for L in PGA_TARGET_LAYERS}
    max_probe = max(probe_per.values())
    rank_now = next(iter(U_k_per_L.values())).shape[1]
    iter_history.append({'iter': it, 'rank': rank_now,
                          'max_probe': max_probe,
                          'probe_per_layer': {str(L): probe_per[L] for L in PGA_TARGET_LAYERS}})
    print(f'  iter {it}: rank={rank_now}, max LOO probe = {max_probe:.3f}  '
          f'(extract {extract_t:.1f}s)')

    if max_probe < ADV_TOL_PROBE:
        print(f'  -> converged: probe below {ADV_TOL_PROBE}')
        converged = True
        break

    # Augment U_k at each layer
    grew = False
    for L in PGA_TARGET_LAYERS:
        Xm = np.vstack([a[L] for a in pga_mem])
        Xc = np.vstack([a[L] for a in pga_clean])
        sc = sc_per_L[L]
        Xm_s = (Xm - sc.mean_) / sc.scale_
        Xc_s = (Xc - sc.mean_) / sc.scale_
        w_new = lr_direction(Xm_s, Xc_s)
        U_k = U_k_per_L[L]
        proj_coef = U_k.T @ w_new
        w_orth = w_new - U_k @ proj_coef
        norm = np.linalg.norm(w_orth)
        if norm < 1e-4:
            continue
        w_orth = w_orth / norm
        U_k_per_L[L] = np.hstack([U_k, w_orth.reshape(-1, 1)])
        grew = True
    if not grew:
        print('  -> no augmentation possible (all directions in current subspace), stopping')
        break

final_rank = next(iter(U_k_per_L.values())).shape[1]
print(f'\nFinal adversarial PGA: rank={final_rank} per layer (target layers {PGA_TARGET_LAYERS})')

Extracting baseline activations for adversarial PGA build ...
  done (0.3s)
  iter 0: rank=1, max LOO probe = 0.791  (extract 0.4s)
  iter 1: rank=2, max LOO probe = 0.646  (extract 0.4s)
  iter 2: rank=3, max LOO probe = 0.558  (extract 0.4s)
  iter 3: rank=4, max LOO probe = 0.483  (extract 0.4s)
  -> converged: probe below 0.55

Final adversarial PGA: rank=4 per layer (target layers [3, 4, 5])


## 4. Install rank-k null projection hooks

In [25]:
def install_adv_pga_hooks(model, U_k_per_L, sc_per_L):
    """Install rank-k null projection hooks at all target layers.
    Hook applies P = I - U_k U_k^T in the per-layer standardised space.
    Returns a list of handles for cleanup."""
    handles = []
    for L in U_k_per_L:
        U = U_k_per_L[L]
        sc = sc_per_L[L]
        P = np.eye(U.shape[0]) - U @ U.T
        proj_fn = make_proj_fn(P, sc.mean_, sc.scale_)
        def make_hook(p_fn):
            def hook(_m, _inp, output):
                x = output[0] if isinstance(output, tuple) else output
                x_out = p_fn(x)
                if isinstance(output, tuple):
                    return (x_out,) + output[1:]
                return x_out
            return hook
        h = model.gpt_neox.layers[L].register_forward_hook(make_hook(proj_fn))
        handles.append(h)
    return handles

def remove_hooks(handles):
    for h in handles:
        h.remove()

print(f'Adversarial PGA hooks ready (rank={final_rank}, layers={PGA_TARGET_LAYERS}).')

Adversarial PGA hooks ready (rank=4, layers=[3, 4, 5]).


## 5. Run lm-evaluation-harness on baseline

In [26]:
def run_eval(model, tokenizer, tasks, batch_size, num_fewshot, limit, tag):
    """Run lm-eval task-by-task. Returns dict {task: accuracy or None}.

    Per-task isolation: a single broken task (e.g. PIQA after a future
    `datasets` upgrade) only loses that one entry instead of aborting
    the entire eval. Failures are reported but do not raise.
    """
    print(f'\n--- Running lm-eval [{tag}] on {len(tasks)} tasks ---')
    lm = HFLM(pretrained=model, tokenizer=tokenizer,
              batch_size=batch_size, device=DEVICE)
    out, failures = {}, {}
    for i, task in enumerate(tasks, 1):
        print(f'  [{tag}] ({i}/{len(tasks)}) {task} ...', flush=True)
        try:
            raw = evaluator.simple_evaluate(
                model=lm, tasks=[task],
                num_fewshot=num_fewshot, limit=limit,
            )
            result = (raw or {}).get('results', {}).get(task, {}) or {}
            score = None
            for k in ['acc,none', 'acc', 'acc_norm,none', 'acc_norm']:
                if k in result:
                    score = float(result[k]); break
            out[task] = score
            if score is None:
                print(f'    ! {task}: ran but no acc/acc_norm metric in result')
        except Exception as e:
            out[task] = None
            failures[task] = f'{type(e).__name__}: {str(e).splitlines()[0][:200]}'
            print(f'    ! {task} FAILED -> {failures[task]}')
    print('  results:', out)
    if failures:
        print('  failures:', failures)
    return out

baseline_results = run_eval(model, tokenizer, TASKS,
                              BATCH_SIZE, NUM_FEWSHOT, LIMIT, 'BASELINE')



--- Running lm-eval [BASELINE] on 5 tasks ---
  [BASELINE] (1/5) hellaswag ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for hellaswag on rank 0...
100%|██████████| 10042/10042 [00:04<00:00, 2307.19it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikeliho

  [BASELINE] (2/5) piqa ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for piqa on rank 0...
100%|██████████| 1838/1838 [00:01<00:00, 1054.32it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikelihood requ

  [BASELINE] (3/5) boolq ...


...

  [BASELINE] (4/5) arc_easy ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for arc_easy on rank 0...
100%|██████████| 2376/2376 [00:02<00:00, 1099.49it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikelihood 

  [BASELINE] (5/5) winogrande ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for winogrande on rank 0...
100%|██████████| 1267/1267 [00:00<00:00, 137894.63it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikelih

  results: {'hellaswag': 0.26618203545110536, 'piqa': 0.5979325353645266, 'boolq': 0.5155963302752293, 'arc_easy': 0.36826599326599324, 'winogrande': 0.5193370165745856}


## 6. Run lm-evaluation-harness with adversarial PGA hooks installed

In [27]:
handles = install_adv_pga_hooks(model, U_k_per_L, sc_per_L)
try:
    pga_results = run_eval(model, tokenizer, TASKS,
                            BATCH_SIZE, NUM_FEWSHOT, LIMIT, 'ADV-PGA')
finally:
    remove_hooks(handles)
    print('  adversarial PGA hooks removed.')


--- Running lm-eval [ADV-PGA] on 5 tasks ---
  [ADV-PGA] (1/5) hellaswag ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for hellaswag on rank 0...
100%|██████████| 10042/10042 [00:03<00:00, 2560.02it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikeliho

  [ADV-PGA] (2/5) piqa ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for piqa on rank 0...
100%|██████████| 1838/1838 [00:01<00:00, 1071.27it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikelihood requ

  [ADV-PGA] (3/5) boolq ...


...

  [ADV-PGA] (4/5) arc_easy ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for arc_easy on rank 0...
100%|██████████| 2376/2376 [00:02<00:00, 846.84it/s] 
INFO:lm-eval:Running loglikelihood requests
Running loglikelihood 

  [ADV-PGA] (5/5) winogrande ...


INFO:lm-eval:Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
INFO:lm-eval:Using pre-initialized model
INFO:lm-eval:`group` and `group_alias` keys in TaskConfigs are deprecated and will be removed in v0.4.5 of lm_eval. The new `tag` field will be used to allow for a shortcut to a group of tasks one does not wish to aggregate metrics across. `group`s which aggregate across subtasks must be only defined in a separate group config file, which will be the official way to create groups that support cross-task aggregation as in `mmlu`. Please see the v0.4.4 patch notes and our documentation: https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/new_task_guide.md#advanced-group-configs for more information.
INFO:lm-eval:Setting fewshot random generator seed to 1234
INFO:lm-eval:Building contexts for winogrande on rank 0...
100%|██████████| 1267/1267 [00:00<00:00, 141654.89it/s]
INFO:lm-eval:Running loglikelihood requests
Running loglikelih

  results: {'hellaswag': 0.2650866361282613, 'piqa': 0.5865070729053319, 'boolq': 0.5443425076452599, 'arc_easy': 0.3552188552188552, 'winogrande': 0.5256511444356748}
  adversarial PGA hooks removed.


## 7. Comparison report

In [28]:
print(f'\n{"task":<14}{"baseline":>12}{"adv-PGA":>14}{"Δacc":>10}{"verdict":>14}')
print('-' * 64)
deltas = {}
for task in TASKS:
    b = baseline_results.get(task)
    p = pga_results.get(task)
    if b is None or p is None:
        print(f'  {task:<14}{"N/A":>12}{"N/A":>14}{"N/A":>10}{"missing":>14}')
        continue
    d = p - b
    deltas[task] = d
    verdict = 'OK' if abs(d) < 0.02 else ('damage' if d < -0.02 else 'gain')
    print(f'  {task:<14}{b:>12.4f}{p:>14.4f}{d:>+10.4f}{verdict:>14}')

mean_delta = float(np.mean(list(deltas.values()))) if deltas else 0.0
print(f'\n  Mean Δacc across {len(deltas)} tasks: {mean_delta:+.4f}')
if abs(mean_delta) < 0.01:
    print('  -> Adversarial PGA preserves capability (mean shift < 1pp)')
elif mean_delta < -0.01:
    print(f'  -> Adversarial PGA damages capability by {abs(mean_delta)*100:.1f}pp on average')
else:
    print(f'  -> Adversarial PGA gives {mean_delta*100:.1f}pp average improvement (likely noise)')


task              baseline       adv-PGA      Δacc       verdict
----------------------------------------------------------------
  hellaswag           0.2662        0.2651   -0.0011            OK
  piqa                0.5979        0.5865   -0.0114            OK
  boolq               0.5156        0.5443   +0.0287          gain
  arc_easy            0.3683        0.3552   -0.0130            OK
  winogrande          0.5193        0.5257   +0.0063            OK

  Mean Δacc across 5 tasks: +0.0019
  -> Adversarial PGA preserves capability (mean shift < 1pp)


## 8. Save results + figure

In [29]:
out = {
    'config': {
        'model': MODEL_NAME, 'tasks': TASKS, 'batch_size': BATCH_SIZE,
        'num_fewshot': NUM_FEWSHOT, 'limit': LIMIT,
        'pga_target_layers': PGA_TARGET_LAYERS,
        'adv_max_iters': ADV_MAX_ITERS, 'adv_tol_probe': ADV_TOL_PROBE,
        'effective_rank': final_rank,
        'n_mem': len(MEM_TEXTS), 'n_clean': len(CLEAN_TEXTS),
    },
    'baseline_results': baseline_results,
    'adv_pga_results': pga_results,
    'deltas': deltas,
    'mean_delta': mean_delta,
    'iter_history': iter_history,
}
out_path = os.path.join(OUT_DIR, 'mldu_e_capability_benchmarks_adversarial.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved {out_path}')

# Figure: bar plot of baseline vs adv-PGA per task, with Δacc annotated
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

ax = axes[0]
x = np.arange(len(TASKS))
w = 0.35
b_vals = [baseline_results.get(t, 0) for t in TASKS]
p_vals = [pga_results.get(t, 0) for t in TASKS]
ax.bar(x - w/2, b_vals, w, label='baseline', color='steelblue')
ax.bar(x + w/2, p_vals, w, label=f'adv-PGA (rank={final_rank})', color='tab:green')
for i, t in enumerate(TASKS):
    d = deltas.get(t)
    if d is not None:
        ax.annotate(f'{d:+.3f}', xy=(i, max(b_vals[i], p_vals[i]) + 0.015),
                    ha='center', fontsize=8,
                    color='red' if abs(d) > 0.02 else 'black')
ax.set_xticks(x); ax.set_xticklabels(TASKS, rotation=20)
ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1.0)
ax.set_title(f'lm-eval-harness zero-shot: baseline vs adversarial PGA (rank={final_rank})')
ax.legend(); ax.grid(alpha=0.3, axis='y')

ax = axes[1]
if iter_history:
    iters = [h['iter'] for h in iter_history]
    ranks = [h['rank'] for h in iter_history]
    max_probes = [h['max_probe'] for h in iter_history]
    ax2 = ax.twinx()
    l1 = ax.plot(iters, max_probes, 'o-', color='tab:red', lw=2, ms=8, label='max LOO probe')
    l2 = ax2.plot(iters, ranks, 's-', color='tab:blue', lw=2, ms=8, label='rank')
    ax.axhline(ADV_TOL_PROBE, color='red', linestyle=':', alpha=0.5,
               label=f'convergence target ({ADV_TOL_PROBE})')
    ax.set_xlabel('Adversarial iteration'); ax.set_ylabel('Max LOO probe across layers',
                                                            color='tab:red')
    ax2.set_ylabel('U_k rank', color='tab:blue')
    ax.set_title('Adversarial PGA convergence trajectory')
    ax.tick_params(axis='y', labelcolor='tab:red')
    ax2.tick_params(axis='y', labelcolor='tab:blue')
    lns = l1 + l2
    labs = [l.get_label() for l in lns]
    ax.legend(lns, labs, loc='upper right'); ax.grid(alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, 'fig_capability_benchmarks_adversarial.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')

Saved /kaggle/working/mldu_e_capability_benchmarks_adversarial.json
Saved /kaggle/working/fig_capability_benchmarks_adversarial.png


In [ ]:
# === Diagnostic: per-item PIQA predictions under baseline / PGA / adv-PGA ===
# Goal: the aggregate PIQA score was identical (0.58651) for PGA-treated and
# adversarial-PGA-treated runs. That can mean either:
#   (a) both interventions truly never flip the argmax on PIQA items, or
#   (b) the hooks weren't actually applied during eval (silent no-op).
# This cell decides between (a) and (b) by inspecting per-item predictions
# on a small PIQA subset under all three conditions, on the SAME model object.
#
# Side-effects: temporarily installs and removes hooks on `model`. Leaves the
# model with no hooks at the end, matching the state assumed by later cells.

import json, numpy as np
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM

DIAG_LIMIT = 100   # ~6% of PIQA val; fast and statistically informative
DIAG_TASK  = 'piqa'

def _extract_pred(sample):
    """Return (pred_idx, gold_idx, [scores]) from a multi-choice sample log."""
    fr = sample.get('filtered_resps') or []
    scores = []
    for entry in fr:
        try:
            if isinstance(entry, (list, tuple)) and entry:
                scores.append(float(entry[0]))
            else:
                scores.append(float(entry))
        except (TypeError, ValueError):
            scores.append(float('nan'))
    pred = int(np.argmax(scores)) if scores and not any(np.isnan(scores)) else None
    gold = sample.get('target')
    try: gold = int(gold)
    except (TypeError, ValueError): pass
    return pred, gold, scores

def _run_piqa(lm, tag):
    raw = evaluator.simple_evaluate(
        model=lm, tasks=[DIAG_TASK],
        num_fewshot=0, limit=DIAG_LIMIT, log_samples=True,
    )
    samples = (raw or {}).get('samples', {}).get(DIAG_TASK, []) or []
    rows = []
    for s in samples:
        pred, gold, scores = _extract_pred(s)
        rows.append({'doc_id': s.get('doc_id'), 'pred': pred,
                     'gold': gold, 'scores': scores})
    correct = [r for r in rows if r['pred'] is not None and r['pred'] == r['gold']]
    acc = (len(correct) / len(rows)) if rows else float('nan')
    print(f'  [{tag}] n={len(rows)}  acc={acc:.4f}')
    return rows

print('=== PIQA per-item diagnostic on the SAME model object ===')
print(f'  task={DIAG_TASK}  limit={DIAG_LIMIT}\n')

lm = HFLM(pretrained=model, tokenizer=tokenizer,
          batch_size=BATCH_SIZE, device=DEVICE)

# 1) baseline (no hooks)
diag_baseline = _run_piqa(lm, 'baseline   ')

# 2) PGA hooks
diag_pga = None
try:
    h_pga = install_pga_hooks(model, pga_directions)
    diag_pga = _run_piqa(lm, 'PGA-treated')
finally:
    try: remove_hooks(h_pga)
    except Exception: pass

# 3) adv-PGA hooks
diag_adv = None
try:
    h_adv = install_adv_pga_hooks(model, U_k_per_L, sc_per_L)
    diag_adv = _run_piqa(lm, 'adv-PGA    ')
finally:
    try: remove_hooks(h_adv)
    except Exception: pass

def _compare(a, b, name_a, name_b):
    if not a or not b:
        print(f'  {name_a} vs {name_b}: skipped (one condition missing)'); return None
    pa = {r['doc_id']: r['pred'] for r in a}
    pb = {r['doc_id']: r['pred'] for r in b}
    common = sorted(set(pa) & set(pb))
    same = sum(1 for d in common if pa[d] == pb[d])
    diff_ids = [d for d in common if pa[d] != pb[d]]
    print(f'  {name_a:<11} vs {name_b:<11}: '
          f'{same}/{len(common)} predictions identical, {len(diff_ids)} differ')
    if 0 < len(diff_ids) <= 15:
        print(f'      differing doc_ids: {diff_ids}')
    return len(diff_ids)

print('\n--- per-item agreement ---')
n_diff_b_pga = _compare(diag_baseline, diag_pga, 'baseline', 'PGA')
n_diff_b_adv = _compare(diag_baseline, diag_adv, 'baseline', 'adv-PGA')
n_diff_pga_adv = _compare(diag_pga, diag_adv, 'PGA', 'adv-PGA')

print('\n--- verdict ---')
if diag_pga and diag_adv:
    if n_diff_pga_adv == 0:
        if n_diff_b_pga == 0 and n_diff_b_adv == 0:
            print('  RED FLAG: all three conditions produce identical predictions '
                  'on every item. Hooks are almost certainly not being applied '
                  'during eval. Inspect score deltas below — if also identical, '
                  'the residual stream is unchanged.')
        else:
            print('  PGA and adv-PGA differ from baseline but match each other '
                  'item-for-item. Possible cause: both interventions project out '
                  'the same direction on PIQA-relevant residual components. '
                  'Worth checking, but not a hooks-not-applied bug.')
    else:
        print(f'  PGA and adv-PGA differ on {n_diff_pga_adv} items. The matching '
              'aggregate score on the full PIQA set was a small-sample '
              'coincidence — hooks ARE active. No action needed.')

# Persist for the paper / later inspection
import os
diag_out = {
    'task': DIAG_TASK, 'limit': DIAG_LIMIT,
    'baseline': diag_baseline,
    'pga': diag_pga,
    'adv_pga': diag_adv,
    'n_diff_baseline_vs_pga': n_diff_b_pga,
    'n_diff_baseline_vs_adv': n_diff_b_adv,
    'n_diff_pga_vs_adv': n_diff_pga_adv,
}
diag_path = os.path.join(OUT_DIR, 'mldu_e_piqa_hook_diagnostic.json')
with open(diag_path, 'w') as f:
    json.dump(diag_out, f, indent=2, default=str)
print(f'\nSaved {diag_path}')


## 9. Interpretation

**What this notebook tests.** The companion notebook
`mldu-e-capability-benchmarks.ipynb` showed that the rank-1 LR-coef PGA
at L5--L6 has mean $\Delta\mathrm{acc} = -0.005$ across HellaSwag,
PIQA, BoolQ, ARC-Easy, and WinoGrande --- essentially no capability
damage. But the rank-1 PGA leaves the cross-sequence signature
recoverable by a re-fitting attacker. \emph{Adversarial PGA} closes
that gap with a rank-$6$ subspace per layer applied at all six layers
simultaneously --- a much more aggressive intervention. Does it still
preserve capability?

**Reading the results.**

* If mean $|\Delta\mathrm{acc}| < 0.01$ across the five tasks, the
  recall-preservation finding from the upgrades-comparison notebook
  generalises: adversarial PGA is genuinely \emph{both} stealthy and
  capability-preserving.
* If $\Delta\mathrm{acc}$ is significantly negative, the recall metric
  was masking deeper damage and the paper should report adversarial
  PGA's capability cost honestly.
* The convergence trajectory panel shows whether the loop converged
  cleanly (max probe drops monotonically below the target) or hit the
  iteration cap.

**Decision rule.**

* Mean $|\Delta\mathrm{acc}| < 0.01$: ADD a one-sentence capability
  preservation claim to the Adversarial PGA paragraph in §7.4 of the
  arxiv version.
* $-0.02 < $ mean $\Delta\mathrm{acc} < -0.01$: report as caveat in
  Limitations.
* mean $\Delta\mathrm{acc} < -0.02$: revisit the claim that
  adversarial PGA is capability-preserving --- the recall metric was
  too narrow.


#### Outputs gallery — `mldu-e-capability-benchmarks-adversarial-v2.ipynb`

Figures and JSON results below were produced by this module's published run.


In [30]:
# === Outputs gallery for mldu-e-capability-benchmarks-adversarial-v2.ipynb ===
# Auto-embedded from MLDU-main/figures/ and MLDU-main/results/
print('Module artifacts:')
print('  fig_capability_benchmarks_adversarial.png')
print('  mldu_e_capability_benchmarks_adversarial_results.json')


Module artifacts:
  fig_capability_benchmarks_adversarial.png
  mldu_e_capability_benchmarks_adversarial_results.json



---

## Module: `mldu-e-vocab-matched-pools.ipynb`

_Vocab-matched pools — token-budget control._


<!-- [reviewer-header] auto-generated; safe to keep at the top of the notebook -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rupawheatly/MLDU/blob/main/notebooks/mldu_e_vocab_matched_pools.ipynb)

## Reviewer notes

**What this notebook does.** Resolves MLDU-E Limitation (iv) — *token-identity leakage at
embedding depth* — by constructing vocabulary-matched memorized vs. clean prefix pairs and
verifying that the cross-sequence probe's L0–L1 spike (currently $0.79$, attributed to surface
vocabulary differences) disappears when the prefixes share their token vocabulary.

**Why this matters.** The paper's PGA robustness check (Appendix Y.6) shows worst-case probe
$0.786$ overall, but $0.71$ at memorization-relevant layers L2–L6. The L0–L1 spike to $0.79$
is currently framed as "a predicted consequence — PGA does not target surface-token
distinguishability". This notebook upgrades that framing: with vocabulary-matched pools,
the spike either disappears (in which case PGA's robustness story is strictly cleaner) or
persists (in which case we have new mechanistic information about embedding-depth
memorization).

**The recipe.** For each natural memorized sequence, we construct $K\!=\!5$ clean variants
that share its vocabulary at the prefix tokens and differ only in a small content fragment.
Example for the Apache License preamble: swap `"Apache"` $\to$ `"Mozilla"` / `"MIT"` /
`"BSD"` / `"Eclipse"` / `"GNU"` while keeping all surrounding tokens identical. The probe
now has to distinguish memorization signal vs. clean signal *without* having access to the
surface vocabulary difference.

**Output.** `mldu_e_vocab_matched_pool_results.json` and
`fig_mldu_e_vocab_matched.png`, both written to the platform working directory.

**Runtime.** ~15–30 min on a Colab/Kaggle T4 (Pythia-70M is fast; GPT-2 medium adds ~10 min).

## 0. Setup

In [31]:
import os, json, random
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
import matplotlib.pyplot as plt

if os.path.exists('/kaggle/working'):
    OUT_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    OUT_DIR = '/content'
print(f'Output directory: {OUT_DIR}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}, PyTorch: {torch.__version__}')

Output directory: /kaggle/working
Device: cuda, PyTorch: 2.10.0+cu128


## 1. Vocabulary-matched memorized + clean pool construction

Each entry has a **memorized prefix**, a **memorized continuation** (what the model recalls),
and a **list of clean continuations** sharing the prefix vocabulary but differing only in
the swappable token. The mem/clean labels are at the *continuation* token level.

In [32]:
# 12 vocabulary-matched mem/clean prefix pools — extended from N=3 to N=12 for
# tighter LOO probe variance bands. Each pool: shared prefix (identical token-for-token
# across mem and clean), one mem continuation (memorized by Pythia per paper R2
# screening), and 5 clean continuations vocabulary-matched at the prefix.
VOCAB_MATCHED_POOLS = [
    {
        'name': 'apache_license',
        'prefix': 'Licensed under the',
        'mem_continuation': ' Apache License, Version 2.0',
        'clean_continuations': [
            ' Mozilla License, Version 2.0',
            ' Eclipse License, Version 1.0',
            ' Boost License, Version 1.0',
            ' Custom License, Version 1.0',
            ' Lambda License, Version 0.5',
        ],
    },
    {
        'name': 'mit_grant',
        'prefix': 'Permission is hereby granted, free of charge, to any person obtaining a copy',
        'mem_continuation': ' of this software and associated documentation files',
        'clean_continuations': [
            ' of this notebook and associated examination scripts',
            ' of this manuscript and associated reference materials',
            ' of this proposal and associated supplementary materials',
            ' of this dataset and associated annotation files',
            ' of this protocol and associated calibration files',
        ],
    },
    {
        'name': 'gpl_header',
        'prefix': 'This program is free software: you can redistribute it and/or modify',
        'mem_continuation': ' it under the terms of the GNU General Public License',
        'clean_continuations': [
            ' it under the terms of the FSF Special Public License',
            ' it under the terms of the BSD General Reference License',
            ' it under the terms of the MIT Generic Standard License',
            ' it under the terms of the EFF General Custom License',
            ' it under the terms of the EU General Service License',
        ],
    },
    {
        'name': 'bsd_redistribution',
        'prefix': 'Redistribution and use in source and binary forms, with or without modification,',
        'mem_continuation': ' are permitted provided that the following conditions are met',
        'clean_continuations': [
            ' are recommended provided that the following directives are followed',
            ' are tolerated provided that the following requirements are addressed',
            ' are accepted provided that the following stipulations are honored',
            ' are documented provided that the following obligations are stated',
            ' are reviewed provided that the following parameters are recorded',
        ],
    },
    {
        'name': 'mpl_subject',
        'prefix': 'Subject to the terms of this License, each Contributor hereby grants',
        'mem_continuation': ' You a world-wide, royalty-free, non-exclusive license',
        'clean_continuations': [
            ' You a country-specific, fee-bearing, exclusive permission',
            ' You a region-limited, royalty-paying, restricted authorization',
            ' You a state-restricted, fee-bearing, conditional clearance',
            ' You a campus-specific, royalty-paying, contingent permit',
            ' You a building-restricted, fee-bearing, supervised approval',
        ],
    },
    {
        'name': 'isc_permission',
        'prefix': 'Permission to use, copy, modify, and/or distribute this software for any purpose',
        'mem_continuation': ' with or without fee is hereby granted',
        'clean_continuations': [
            ' with or without notice is herewith approved',
            ' with or without payment is herein authorized',
            ' with or without consent is herein blessed',
            ' with or without warning is hereunder permitted',
            ' with or without record is herein acknowledged',
        ],
    },
    {
        'name': 'eclipse_distribution',
        'prefix': 'This Source Code Form is subject to the terms of the',
        'mem_continuation': ' Eclipse Public License, v. 2.0',
        'clean_continuations': [
            ' Lambda Public License, v. 2.0',
            ' Stellar Public License, v. 2.0',
            ' Vector Public License, v. 2.0',
            ' Quantum Public License, v. 2.0',
            ' Compass Public License, v. 2.0',
        ],
    },
    {
        'name': 'cc_preamble',
        'prefix': 'This work is licensed under a Creative',
        'mem_continuation': ' Commons Attribution 4.0 International License',
        'clean_continuations': [
            ' Designs Attribution 4.0 International License',
            ' Sciences Attribution 4.0 International License',
            ' Institute Attribution 4.0 International License',
            ' Council Attribution 4.0 International License',
            ' Society Attribution 4.0 International License',
        ],
    },
    {
        'name': 'lgpl_header',
        'prefix': 'This library is free software; you can redistribute it and/or modify it under the terms of the',
        'mem_continuation': ' GNU Lesser General Public License',
        'clean_continuations': [
            ' BSD Lesser General Public License',
            ' MIT Lesser General Public License',
            ' EFF Lesser General Public License',
            ' EU Lesser General Public License',
            ' ACM Lesser General Public License',
        ],
    },
    {
        'name': 'as_is_disclaimer',
        'prefix': 'THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,',
        'mem_continuation': ' EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO',
        'clean_continuations': [
            ' WRITTEN OR INFORMAL, INCLUDING BUT NOT LIMITED TO',
            ' SIGNED OR INFORMAL, INCLUDING BUT NOT LIMITED TO',
            ' STATED OR INFORMAL, INCLUDING BUT NOT LIMITED TO',
            ' VERBAL OR INFORMAL, INCLUDING BUT NOT LIMITED TO',
            ' POSTED OR INFORMAL, INCLUDING BUT NOT LIMITED TO',
        ],
    },
    {
        'name': 'python_shebang',
        'prefix': '#!/usr/bin/env python3',
        'mem_continuation': '\n# -*- coding: utf-8 -*-',
        'clean_continuations': [
            '\n# -*- coding: latin-1 -*-',
            '\n# -*- coding: ascii-7 -*-',
            '\n# -*- coding: utf-7 -*-',
            '\n# -*- coding: cp1251 -*-',
            '\n# -*- coding: koi8-r -*-',
        ],
    },
    {
        'name': 'copyright_notice',
        'prefix': 'Copyright (c) 2024,',
        'mem_continuation': ' All rights reserved.',
        'clean_continuations': [
            ' Most rights reserved.',
            ' Many rights reserved.',
            ' Some rights reserved.',
            ' Few rights reserved.',
            ' Other rights reserved.',
        ],
    },
]

MODELS_TO_TEST = [
    {'hf_name': 'EleutherAI/pythia-70m', 'depths': list(range(7))},  # 6 transformer layers + embedding
    # Uncomment to also run GPT-2 medium (~10 min extra):
    # {'hf_name': 'gpt2-medium', 'depths': list(range(0, 25))},
]

PROBE_C = 1.0
MAX_ITER = 2000

print(f'Memorized prefixes to test: {len(VOCAB_MATCHED_POOLS)}')
print(f'Clean alternatives per memorized: {len(VOCAB_MATCHED_POOLS[0]["clean_continuations"])}')
print(f'Models to evaluate: {[m["hf_name"] for m in MODELS_TO_TEST]}')


Memorized prefixes to test: 12
Clean alternatives per memorized: 5
Models to evaluate: ['EleutherAI/pythia-70m']


## 2. Activation extraction across all depths

For each (model, sequence) pair, capture residual stream activations at every depth in a
single forward pass. Returns `{depth: (T, D) numpy}` where T is the token count of the
*full prefix+continuation* sequence.

In [33]:
def get_layers(model):
    """Locate the transformer block list (Pythia and GPT-2 use different attribute names)."""
    if hasattr(model, 'transformer') and hasattr(model.transformer, 'h'):
        return model.transformer.h  # GPT-2
    if hasattr(model, 'gpt_neox') and hasattr(model.gpt_neox, 'layers'):
        return model.gpt_neox.layers  # Pythia (GPT-NeoX)
    raise ValueError(f'Unknown model layout: {type(model).__name__}')

def get_embed_module(model):
    """Locate the token-embedding module (paper convention: depth 0 = embedding)."""
    if hasattr(model, 'transformer') and hasattr(model.transformer, 'wte'):
        return model.transformer.wte  # GPT-2 word-token embeddings
    if hasattr(model, 'gpt_neox') and hasattr(model.gpt_neox, 'embed_in'):
        return model.gpt_neox.embed_in  # Pythia
    raise ValueError(f'Unknown embedding location: {type(model).__name__}')

@torch.no_grad()
def extract_all_depths(model, tokenizer, text, depths):
    """Single forward pass; capture residual stream at each depth in `depths`.
    Returns {depth: (T, D) numpy}.
    Convention: depth 0 = embedding output (paper's L0); depth d > 0 = output of
    transformer layer index d-1 (paper's Ld)."""
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(DEVICE)
    captured = {}
    layers = get_layers(model)
    handles = []
    # BUG FIX 1: capture embedding via forward hook on the embedding module (correct 3-arg signature)
    if 0 in depths:
        embed_mod = get_embed_module(model)
        def hook_embed(_m, _inp, output):
            captured[0] = output.detach()
        handles.append(embed_mod.register_forward_hook(hook_embed))
    # Capture each transformer layer output
    for d in depths:
        if d == 0:
            continue
        layer_idx = d - 1
        if layer_idx < 0 or layer_idx >= len(layers):
            continue
        def make_hook(dd):
            def hook(_m, _inp, output):
                x = output[0] if isinstance(output, tuple) else output
                captured[dd] = x.detach()
            return hook
        handles.append(layers[layer_idx].register_forward_hook(make_hook(d)))
    try:
        model(**enc)
    finally:
        for h in handles:
            h.remove()
    return {d: captured[d].squeeze(0).cpu().float().numpy() for d in depths if d in captured}


## 3. Per-token labelling: which positions count as "mem" vs "clean"

Only the **continuation tokens** (after the shared prefix) carry the memorization signal.
We label those tokens accordingly and ignore the prefix tokens (they are identical across
mem and clean by construction).

In [34]:
def get_continuation_token_indices(tokenizer, prefix, full_text):
    """Return the slice [start:end] of token indices in `full_text` that belong to the
    continuation. Asserts that `prefix` tokenizes as a prefix of `full_text` (BPE safety)."""
    prefix_ids = tokenizer(prefix, return_tensors='pt').input_ids[0]
    full_ids = tokenizer(full_text, return_tensors='pt').input_ids[0]
    # BUG FIX 3: verify token-level prefix relationship (BPE boundary safety)
    n_pref = len(prefix_ids)
    if not (full_ids[:n_pref] == prefix_ids).all():
        # Tokenization is not a clean concatenation; find the longest common prefix
        common = 0
        for i in range(min(n_pref, len(full_ids))):
            if prefix_ids[i] != full_ids[i]:
                break
            common += 1
        return common, len(full_ids)
    return n_pref, len(full_ids)

def build_pool_for_depth(model, tokenizer, pool, depths):
    """For one VOCAB_MATCHED_POOLS entry, return per-depth dicts of mem and clean activations.
    Restricted to continuation tokens only (vocabulary-matched comparison)."""
    mem_full = pool['prefix'] + pool['mem_continuation']
    clean_fulls = [pool['prefix'] + c for c in pool['clean_continuations']]
    
    mem_acts_per_depth = extract_all_depths(model, tokenizer, mem_full, depths)
    s_mem, e_mem = get_continuation_token_indices(tokenizer, pool['prefix'], mem_full)
    
    clean_acts_per_depth_per_text = []
    for ct in clean_fulls:
        acts = extract_all_depths(model, tokenizer, ct, depths)
        s_c, e_c = get_continuation_token_indices(tokenizer, pool['prefix'], ct)
        clean_acts_per_depth_per_text.append((acts, s_c, e_c))
    
    out = {}
    for d in depths:
        if d not in mem_acts_per_depth: continue
        mem_d = mem_acts_per_depth[d][s_mem:e_mem]                # (T_cont, D)
        clean_d_list = []
        for acts, s_c, e_c in clean_acts_per_depth_per_text:
            if d in acts:
                clean_d_list.append(acts[d][s_c:e_c])
        if not clean_d_list: continue
        clean_d = np.vstack(clean_d_list)
        out[d] = (mem_d, clean_d)
    return out


## 4. Probe training (LOO with balanced pools, balanced accuracy)

For each depth, fit a linear probe to separate mem vs clean activations. Subsample to
balanced pool sizes; report `balanced_accuracy_score` so chance = 0.50.

In [35]:
def fit_probe_at_depth(mem_acts, clean_acts, seed=42, train_frac=0.7):
    rng = np.random.default_rng(seed)
    n_bal = min(len(mem_acts), len(clean_acts))
    if n_bal < 2:
        return None  # too few samples
    mem_idx = rng.choice(len(mem_acts), n_bal, replace=False)
    clean_idx = rng.choice(len(clean_acts), n_bal, replace=False)
    Xm = mem_acts[mem_idx]
    Xc = clean_acts[clean_idx]
    X = np.vstack([Xm, Xc])
    y = np.concatenate([np.ones(n_bal), np.zeros(n_bal)])
    perm = rng.permutation(len(X)); X, y = X[perm], y[perm]
    n_tr = max(int(len(X) * train_frac), 2)
    if len(X) - n_tr < 2:
        return None
    sc = StandardScaler().fit(X[:n_tr])
    clf = LogisticRegression(C=PROBE_C, max_iter=MAX_ITER, random_state=seed,
                             class_weight='balanced')
    clf.fit(sc.transform(X[:n_tr]), y[:n_tr])
    yp = clf.predict(sc.transform(X[n_tr:]))
    return float(balanced_accuracy_score(y[n_tr:], yp))

## 5. Full sweep: per model × per pool × per depth

In [36]:
all_results = {}

for model_cfg in MODELS_TO_TEST:
    name = model_cfg['hf_name']
    depths = model_cfg['depths']
    print(f'\n=== {name} ===')
    print('  loading model ...')
    tokenizer = AutoTokenizer.from_pretrained(name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(name).to(DEVICE)
    model.eval()
    
    per_pool_per_depth = {}
    for pool in VOCAB_MATCHED_POOLS:
        print(f'  pool: {pool["name"]}')
        depth_data = build_pool_for_depth(model, tokenizer, pool, depths)
        per_depth_acc = {}
        for d in depths:
            if d not in depth_data: continue
            mem_d, clean_d = depth_data[d]
            acc = fit_probe_at_depth(mem_d, clean_d)
            if acc is None: continue
            per_depth_acc[d] = acc
        per_pool_per_depth[pool['name']] = per_depth_acc
        for d in sorted(per_depth_acc):
            print(f'    depth {d}: probe = {per_depth_acc[d]:.3f}')
    
    # Aggregate across pools at each depth (mean of per-pool probe accs)
    aggregate = {}
    for d in depths:
        accs = [per_pool_per_depth[p][d] for p in per_pool_per_depth
                if d in per_pool_per_depth[p]]
        if accs:
            aggregate[d] = {'mean': float(np.mean(accs)), 'max': float(np.max(accs)),
                            'n_pools': len(accs)}
    
    all_results[name] = {'per_pool': per_pool_per_depth, 'aggregate': aggregate}
    
    # Free GPU memory
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print('\nDone.')


=== EleutherAI/pythia-70m ===
  loading model ...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

  pool: apache_license
    depth 0: probe = 0.000
    depth 1: probe = 0.625
    depth 2: probe = 0.750
    depth 3: probe = 0.875
    depth 4: probe = 0.875
    depth 5: probe = 0.750
    depth 6: probe = 1.000
  pool: mit_grant
    depth 0: probe = 0.250
    depth 1: probe = 0.875
    depth 2: probe = 0.875
    depth 3: probe = 0.625
    depth 4: probe = 0.875
    depth 5: probe = 0.750
    depth 6: probe = 0.875
  pool: gpl_header
    depth 0: probe = 0.600
    depth 1: probe = 0.600
    depth 2: probe = 0.700
    depth 3: probe = 0.700
    depth 4: probe = 0.700
    depth 5: probe = 0.700
    depth 6: probe = 0.700
  pool: bsd_redistribution
    depth 0: probe = 0.375
    depth 1: probe = 0.625
    depth 2: probe = 0.875
    depth 3: probe = 0.750
    depth 4: probe = 0.750
    depth 5: probe = 0.875
    depth 6: probe = 0.750
  pool: mpl_subject
    depth 0: probe = 0.425
    depth 1: probe = 0.350
    depth 2: probe = 0.650
    depth 3: probe = 0.775
    depth 4: probe = 0.750
  

## 6. Compare to original (vocabulary-mismatched) results

Report the per-depth probe accuracies for each model. If vocabulary-matched pools eliminate
the L0–L1 spike, mean probe accuracy at those depths should drop close to chance ($0.50$).

**Reference values from the paper** (vocabulary-mismatched pools):
- Pythia-70M: L0=0.79, L1=0.71, L2-L6 below 0.71 (paper Table 28)
- Goal: with vocabulary-matched pools, L0/L1 should drop to ~0.50.

In [37]:
print(f'{"model":<32}{"depth":<8}{"mean probe":<12}{"max probe":<12}{"interpretation":<30}')
print('-' * 100)
for name, res in all_results.items():
    for d in sorted(res['aggregate']):
        a = res['aggregate'][d]
        if a['mean'] < 0.55:
            interp = '✓ near chance — vocab leakage gone'
        elif a['mean'] < 0.65:
            interp = '~ residual leakage remains'
        else:
            interp = '✗ probe still distinguishes'
        depth_label = f'L{d}'  # paper convention: L0=embedding, Ld=output of layer d-1
        print(f'{name:<32}{depth_label:<8}{a["mean"]:<12.3f}{a["max"]:<12.3f}{interp:<30}')

model                           depth   mean probe  max probe   interpretation                
----------------------------------------------------------------------------------------------------
EleutherAI/pythia-70m           L0      0.356       0.625       ✓ near chance — vocab leakage gone
EleutherAI/pythia-70m           L1      0.590       1.000       ~ residual leakage remains    
EleutherAI/pythia-70m           L2      0.727       1.000       ✗ probe still distinguishes   
EleutherAI/pythia-70m           L3      0.640       0.875       ~ residual leakage remains    
EleutherAI/pythia-70m           L4      0.722       1.000       ✗ probe still distinguishes   
EleutherAI/pythia-70m           L5      0.668       0.875       ✗ probe still distinguishes   
EleutherAI/pythia-70m           L6      0.656       1.000       ✗ probe still distinguishes   


## 7. Save results + figure

In [38]:
out = {
    'method': 'Vocabulary-matched cross-sequence LOO probe',
    'description': ('Construct vocabulary-matched mem/clean prefix pairs to eliminate '
                    'embedding-depth surface-token leakage. Compare probe accuracy at L0–L1 '
                    'against vocabulary-mismatched pools (paper Appendix Y.6).'),
    'pools': [{'name': p['name'], 'prefix': p['prefix'],
               'mem_continuation': p['mem_continuation'],
               'n_clean_alternates': len(p['clean_continuations'])}
              for p in VOCAB_MATCHED_POOLS],
    'results': all_results,
}
out_path = os.path.join(OUT_DIR, 'mldu_e_vocab_matched_pool_results.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved {out_path}')

# Figure: per-depth probe accuracy across pools, one line per model
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
for name, res in all_results.items():
    depths = sorted(res['aggregate'].keys())
    means = [res['aggregate'][d]['mean'] for d in depths]
    maxs = [res['aggregate'][d]['max'] for d in depths]
    depth_labels = [f'L{d}' for d in depths]  # paper convention: L0=embedding, Ld=output of layer d-1
    ax.plot(depths, means, 'o-', label=f'{name} (mean across {len(VOCAB_MATCHED_POOLS)} pools)', lw=2, ms=8)
    ax.fill_between(depths, means, maxs, alpha=0.2)
ax.axhline(0.50, color='gray', linestyle='--', alpha=0.6, label='chance')
ax.axhline(0.79, color='red', linestyle=':', alpha=0.6, label='paper L0/L1 spike (vocab-mismatched)')
ax.set_xlabel('Residual stream depth')
ax.set_ylabel('Cross-sequence probe balanced-accuracy')
ax.set_title('Vocabulary-matched mem/clean pools: does the embedding-depth probe spike disappear?')
ax.set_ylim(0.30, 1.05)
ax.legend(loc='best')
ax.grid(alpha=0.3)
fig_path = os.path.join(OUT_DIR, 'fig_mldu_e_vocab_matched.png')
plt.tight_layout()
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')

Saved /kaggle/working/mldu_e_vocab_matched_pool_results.json
Saved /kaggle/working/fig_mldu_e_vocab_matched.png


## 8. Interpretation

**If L0–L1 probe accuracy drops to ~0.50 with vocabulary-matched pools:**
Limitation (iv) is *resolved*. The original L0–L1 spike to $0.79$ in Appendix Y.6 was
purely surface-vocabulary leakage; once we control for it, PGA's robustness story is
clean across all depths. Add to paper: *"With vocabulary-matched memorized/clean pools,
the L0–L1 probe spike disappears (probe ≤ 0.55), confirming that the original spike was
an artifact of prefix-vocabulary mismatch, not a robustness failure of PGA."*

**If L0–L1 stays high even with vocabulary-matched pools:**
We have new mechanistic information — the embedding-depth memorization signal extends
beyond surface-vocabulary differences. This becomes a finding rather than an artifact;
the paper would need to address it (likely by extending PGA to embedding depth).

**Either way:** this experiment converts Limitation (iv) from a hand-wave to a tested
claim, which is exactly what a reviewer is looking for.


#### Outputs gallery — `mldu-e-vocab-matched-pools.ipynb`

Figures and JSON results below were produced by this module's published run.


In [39]:
# === Outputs gallery for mldu-e-vocab-matched-pools.ipynb ===
# Auto-embedded from MLDU-main/figures/ and MLDU-main/results/
print('Module artifacts:')
print('  fig_mldu_e_vocab_matched.png')
print('  mldu_e_vocab_matched_results.json')


Module artifacts:
  fig_mldu_e_vocab_matched.png
  mldu_e_vocab_matched_results.json



---

## Module: `MLDU_E_register_control.ipynb`

_Register control._


<!-- [reviewer-header] auto-generated; safe to keep at the top of the notebook -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rupawheatly/MLDU/blob/main/notebooks/MLDU_E_register_control.ipynb)

## Reviewer notes

**What this notebook does.** **Experiment 1 (post-submission control).** Tests whether the cross-sequence probe reads memorization or formal-register English. Trains the probe on memorized legal text, evaluates on 19 register-matched but unmemorized synthetic licenses.

**Paper section.** §3.2 (cited inline) and Appendix G (Register-vs-Memorization Control subsection)

**Outputs.** 1 JSON (`mldu_e_register_control.json`).

**Hardware / runtime.** T4 GPU, ~~5 min.

**How to run from a fresh GitHub clone.**

1. Click the "Open in Colab" badge above (or upload to Kaggle / run locally).
2. The first code cell installs all dependencies via `pip`.
3. Output paths auto-detect the runtime: Colab Drive (`/content/drive/MyDrive/MIDU/`), Kaggle (`/kaggle/working/`), or a local `./mldu_e_work/` directory. No manual setup is required if you accept the defaults.
4. Mistral-7B notebooks additionally need an `HF_TOKEN` (Colab → Secrets, Kaggle → Add-ons → Secrets, or `os.environ['HF_TOKEN']` locally).

---


# Experiment 1 — Register-vs-memorization control

**Purpose.** Address the biggest unresolved confound in the cross-sequence probe story: is the LOO probe reading *memorization*, or is it reading *formal-register English / legalese*?

**Protocol.** Train the LOO probe on memorized legal sequences (canonical pool from the parent paper). Then evaluate it — *without retraining* — on:

1. Held-out memorized legal text (sanity)
2. **Unmemorized legal text** (the critical test): structurally legalese but absent from pretraining
3. Code + lorem ipsum (existing negative controls)

**Decision rule.**
- Probe accuracy on unmemorized legal ≤ 0.6 (near pure baseline) → memorization-specific signature confirmed.
- Probe accuracy ∈ [0.6, 0.85] → partial register confound; reframe as enriched.
- Probe accuracy ≥ 0.85 → confound dominates; major reframe required.

**Runs on:** Colab (Drive auto-mount) + Kaggle (T4/P100) + local. Detects environment automatically.

**Default architecture:** GPT-2-medium (the parent paper's strongest natural-memorization signal). Optional Pythia-70M.

**Outputs.** `MLDU_E/artifacts/mldu_e_register_control.json` with per-pool LOO accuracy, per-layer trajectories, and a summary verdict.


## 0. Install + setup

In [40]:
!pip install -q transformers accelerate scikit-learn

In [41]:
# === environment detection: works on Colab + Kaggle + local ===
import os, sys, json, math, time, gc, random, warnings
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=UserWarning)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42); random.seed(42)

def detect_env():
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return 'kaggle'
    try:
        import google.colab  # noqa: F401
        return 'colab'
    except ImportError:
        return 'local'

ENV = detect_env()
print(f'env: {ENV}   device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}   '
          f'free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# Standardized paths — match conventions in the existing MLDU-E notebooks:
#   Colab : /content/drive/MyDrive/MIDU/MLDU_E/artifacts (matches MLDU_E_pga_pythia70m_full.ipynb)
#   Kaggle: /kaggle/working/artifacts                    (matches MLDU_E_mia_comparison_kaggle.ipynb)
#   local : ./mldu_e_work/artifacts
if ENV == 'kaggle':
    INPUT = Path('/kaggle/input/midu-mldu-e')   # optional dataset attach
    WORK  = Path('/kaggle/working')
    ART   = WORK / 'artifacts'
    ADP   = WORK / 'MLDU_E' / 'adapters'
elif ENV == 'colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        WORK = Path('/content/drive/MyDrive/MIDU')
    except Exception as e:
        print(f'(drive mount failed: {e}; falling back to /content)')
        WORK = Path('/content')
    INPUT = WORK
    ART   = WORK / 'MLDU_E' / 'artifacts'
    ADP   = WORK / 'MLDU_E' / 'adapters'

ART.mkdir(parents=True, exist_ok=True)
ADP.mkdir(parents=True, exist_ok=True)
print(f'INPUT={INPUT}\nWORK ={WORK}\nART  ={ART}\nADP  ={ADP}')

# HF token: try Kaggle Secrets, then Colab userdata, then env var
HF_TOKEN = None
if ENV == 'kaggle':
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
elif ENV == 'colab':
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN')
if HF_TOKEN: os.environ['HF_TOKEN'] = HF_TOKEN
print(f'HF_TOKEN: {"set" if HF_TOKEN else "(none — Mistral cell will be skipped)"}')


env: kaggle   device: cuda
GPU: Tesla T4   free VRAM: 15.0 GB
INPUT=/kaggle/input/midu-mldu-e
WORK =/kaggle/working
ART  =/kaggle/working/artifacts
ADP  =/kaggle/working/MLDU_E/adapters
HF_TOKEN: (none — Mistral cell will be skipped)


## 1. Embedded sequence pools (memorized + unmemorized legal + controls)

In [42]:
# Memorized legal/license pool (canonical — same as parent paper).
MEMORIZED_LEGAL = [
    'Permission is hereby granted, free of charge, to any person obtaining a copy of this software '
    'and associated documentation files (the "Software"), to deal in the Software without '
    'restriction, including without limitation the rights to use, copy, modify, merge, publish, '
    'distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the '
    'Software is furnished to do so, subject to the following conditions:',

    'GNU GENERAL PUBLIC LICENSE Version 2, June 1991 Copyright (C) 1989, 1991 Free Software '
    'Foundation, Inc., 51 Franklin Street, Fifth Floor, Boston, MA 02110-1301 USA Everyone is '
    'permitted to copy and distribute verbatim copies of this license document, but changing it is '
    'not allowed.',

    'Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file '
    'except in compliance with the License. You may obtain a copy of the License at '
    'http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in '
    'writing, software distributed under the License is distributed on an "AS IS" BASIS,',

    'Redistribution and use in source and binary forms, with or without modification, are permitted '
    'provided that the following conditions are met: 1. Redistributions of source code must retain '
    'the above copyright notice, this list of conditions and the following disclaimer.',

    'This program is free software: you can redistribute it and/or modify it under the terms of the '
    'GNU General Public License as published by the Free Software Foundation, either version 3 of '
    'the License, or (at your option) any later version.',

    'The above copyright notice and this permission notice shall be included in all copies or '
    'substantial portions of the Software. THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF '
    'ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,',

    'This work is licensed under the Creative Commons Attribution-ShareAlike 4.0 International '
    'License. To view a copy of this license, visit http://creativecommons.org/licenses/by-sa/4.0/ '
    'or send a letter to Creative Commons, PO Box 1866, Mountain View, CA 94042, USA.',
]

# UNMEMORIZED LEGAL pool — paraphrased / synthetically rewritten legalese designed to be
# *register-matched* but *not present in pretraining*. Each item is structurally legalese
# (NOT a real license; deliberately rewritten so models will not have memorized it) but
# semantically a license-style passage. Screen these on the model in cell 4 to confirm
# log P/tok well below memorized.
UNMEMORIZED_LEGAL = [
    'Subject to the conditions enumerated below, the Grantor extends to any natural or artificial '
    'person who lawfully acquires a duplicate of this codebase a non-revocable authorization to '
    'inspect, transform, recombine, package, and onward-distribute the codebase or fragments thereof, '
    'provided that the original notices remain affixed to all derived artifacts.',

    'PUBLIC OPEN-WORK LICENCE Edition 4, March 2014 Copyright (Q) 2009-2014 Open-Work Trust, 12 '
    'Carrington Lane, Wellington 6011 New Zealand. Verbatim reproduction and onward conveyance of '
    'this licence document are explicitly authorized; alteration of the licence wording is '
    'expressly forbidden.',

    'Issued under the Reciprocal Public Distribution Permit, Edition 1.3 (the "Permit"); the '
    'Licensee shall not invoke the present codebase outside the boundaries of the Permit. A current '
    'copy of the Permit is maintained at https://example.org/rpd/1.3 and is binding on all '
    'derivative redistributions.',

    'Onward conveyance and adaptation in unprocessed or compiled form, with or without amendment, '
    'are sanctioned where the following stipulations are jointly satisfied: (i) the upstream notice '
    'of authorship is preserved verbatim; (ii) the Permit URL appears in machine-readable metadata.',

    'The present artifact is offered as unencumbered software: the recipient may further convey or '
    'amend it under the conditions of the Reciprocal Open Conveyance Charter, in either the third '
    'or any subsequent revision, at the recipient discretion.',

    'The aforementioned attribution clause and the present permission clause must be reproduced in '
    'their entirety within all forwarded copies and within any non-trivial derivative thereof. THIS '
    'ARTIFACT IS DELIVERED ON A NO-WARRANTY FOOTING; NO WARRANTY OF FITNESS, MERCHANTABILITY, OR '
    'NON-INFRINGEMENT IS EXTENDED.',

    'This contribution is offered under the Open Reciprocity Charter, Edition 5.1. To examine the '
    'present text of the Charter, consult https://example.org/orc/5.1 or write to the Open '
    'Reciprocity Trust, PO Box 4017, Christchurch 8140, Aotearoa New Zealand.',

    'BILATERAL CODE-SHARING COMPACT Revision 7.2, October 2018. Copyright (X) 2014-2018 Bilateral '
    'Code Stewardship Council, Suite 318, 482 Maritime Avenue, Halifax NS B3J 2K9 Canada. Authorized '
    'parties may reproduce, redistribute, or recompose verbatim instances of this Compact, but no '
    'partial alteration of operative clauses is sanctioned.',

    'Pursuant to the Cooperative Source Stewardship Accord (Edition 2.4), the Custodian hereby '
    'confers upon any party that lawfully obtains a faithful instance of the present codebase a '
    'perpetual, transferable, but revocable warrant to examine, modify, recombine, package, and '
    'further distribute the same, conditional on continued observance of the operative clauses.',

    'The present allotment is provided without express or implied warranty of any character, and '
    'the Distributor disclaims all liability arising from any consequential, incidental, or special '
    'damages occasioned by the recipient operation, integration, or reliance upon the allotment, '
    'irrespective of whether the Distributor was placed on notice of such possibility.',

    'INTERNATIONAL DEVELOPER ENCLAVE PERMIT, Subscription Tier C, calendar year 2017. Holders of '
    'this Tier C subscription are granted, subject to clauses (a) through (m) inclusive, an exclusive '
    'right of installation upon authorized hardware located within the Permitted Territory, and a '
    'concurrent right of incidental archival reproduction.',

    'Forwarding and modification, in either pre-compilation or post-compilation form, accompanied or '
    'unaccompanied by emendation, are sanctioned where the following composite criteria obtain: (a) '
    'the antecedent attribution legend is reproduced verbatim; (b) the present permission text is '
    'incorporated into all derived artifacts without abridgment.',

    'The Foundation grants to each lawful recipient of this codebase, on a perpetual non-exclusive '
    'basis, a transferable warrant of reproduction, recompilation, and onward conveyance, contingent '
    'on the recipient ongoing observance of the obligations enumerated in Schedule II hereto and '
    'the maintenance of unaltered upstream attribution notices.',

    'CIVIC OPEN INSTRUMENT, Editorial Cycle 11. Copyright (R) 2011-2019 Civic Open Custodianship, '
    'Avenue de la Recherche 184, 1050 Brussels, Belgium. Custodianship is exercised on behalf of the '
    'aggregated contributor cohort. Verbatim onward conveyance of this Instrument is authorized; '
    'alteration of operative wording is interdicted.',

    'The undersigned Issuer, in consideration of the mutual covenants set forth herein, conveys to '
    'each holder a non-revocable perpetual entitlement to operate, replicate, transform, and '
    'further convey the present codebase, contingent only on the holder continuing observance of '
    'the attribution and notice-preservation requirements.',

    'NO PORTION OF THE PRESENT ALLOTMENT, NOR ANY DERIVATIVE THEREOF, MAY BE PUT TO COMMERCIAL USE '
    'WITHOUT THE PRIOR WRITTEN CONSENT OF THE CUSTODIAN; ANY UNAUTHORIZED COMMERCIAL EXPLOITATION '
    'CONSTITUTES A MATERIAL BREACH OF THE OPERATIVE STIPULATIONS AND MAY GIVE RISE TO COMPENSATORY '
    'AND INJUNCTIVE REMEDIES IN FAVOR OF THE CUSTODIAN.',

    'Distribution of this codebase, whether in originating or in transformed condition, must be '
    'accompanied by an unaltered facsimile of the present permission text, the antecedent '
    'attribution legend, and any subsidiary notices appended thereto by intermediate distributors '
    'in the chain of conveyance.',

    'Pursuant to the operative provisions of the Composite Contribution Permit (Revision 6.0), the '
    'recipient is barred from removing, defacing, or otherwise interfering with any attribution, '
    'notice, or annotation affixed to the codebase by upstream contributors, and shall propagate '
    'all such markings into any derivative artifact without abridgment.',

    'COMMERCIAL INVOCATION OF THE PRESENT ARTIFACT IS PERMITTED ONLY UPON EXECUTION OF A SUPPLEMENTAL '
    'COMMERCIAL TERMS RIDER WITH THE CUSTODIAN; ABSENT SUCH RIDER, THE RECIPIENT WARRANT IS RESTRICTED '
    'TO NON-COMMERCIAL ACADEMIC, RESEARCH, AND PERSONAL USE, AND DOES NOT EXTEND TO RESALE OR '
    'INCORPORATION INTO REVENUE-GENERATING SERVICES.',

    'The Custodian reserves the right to publish, from time to time, revised editions of the present '
    'Charter; revised editions shall not retroactively alter warrants conveyed under preceding '
    'editions, but shall govern any conveyance occurring on or after the effective date of the '
    'revised edition as designated in the official record of editions.',
]

# Matched-prefix decoys for memorized — same protocol as parent paper:
# share the first ~10 tokens, randomize the rest from the same register.
CLEAN_LEGAL = [
    'Permission is hereby granted, free of charge, to any person willing to undertake the following '
    'voluntary maintenance duties on the wikipedia article corpus during the calendar year, including '
    'occasional rewrites, citation chasing, and seasonal vandalism reverts as conditions allow.',

    'GNU GENERAL PUBLIC LICENSE Version 2, June 1991 Subscribers and prospective subscribers are '
    'invited to attend the quarterly community potluck dinner held at the cooperative meeting hall '
    'on the second Saturday of every third month from seven through ten in the evening.',

    'Licensed under the Apache License, Version 2.0 (the "License"); each ticketed attendee is '
    'permitted one complimentary beverage and one entrée selection from the seasonally rotating '
    'tasting menu prepared by the visiting chef during the festival weekend programming.',

    'Redistribution and use in source and binary forms, with or without modification, of the annual '
    'community garden plot allocation are governed by the rotating waiting list maintained by the '
    'volunteer steering committee on a strictly first-come basis throughout the planting season.',

    'This program is free software: you can redistribute it and/or arrange to host the visiting '
    'string quartet for one of the eight scheduled chamber music recitals during the upcoming '
    'concert season at the small auditorium on the second floor of the public library annex.',

    'The above copyright notice and this permission notice shall be included in all copies of the '
    'forthcoming community newsletter, which goes to print on the first Friday of each month and '
    'is delivered by volunteer carriers to subscribers across the four neighborhood districts.',

    'This work is licensed under the Creative Commons Attribution-ShareAlike 4.0 International '
    'standard for documenting amateur radio field-day exercises conducted by the regional club '
    'during the third weekend of June each year, with logs submitted within fourteen calendar days.',
]

# Code + placeholder controls (cluster-specific negative classes per parent paper).
CODE_CONTROLS = [
    'def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[len(arr) // 2]\n'
    '    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n'
    '    right = [x for x in arr if x > pivot]\n    return quicksort(left) + middle + quicksort(right)',

    'class BinaryTree:\n    def __init__(self, value):\n        self.value = value\n'
    '        self.left = None\n        self.right = None\n    def insert(self, value):\n'
    '        if value < self.value:\n            if self.left is None:\n                self.left = BinaryTree(value)',

    'def fibonacci(n):\n    if n <= 1:\n        return n\n    a, b = 0, 1\n    for _ in range(n - 1):\n'
    '        a, b = b, a + b\n    return b\n\nresult = fibonacci(20)\nprint(f"fib(20) = {result}")',

    'import json\nfrom pathlib import Path\n\ndef load_config(path: str) -> dict:\n'
    '    cfg_path = Path(path)\n    if not cfg_path.exists():\n        raise FileNotFoundError(path)\n'
    '    return json.loads(cfg_path.read_text(encoding="utf-8"))',

    'class RateLimiter:\n    def __init__(self, max_calls: int, period: float):\n'
    '        self.max_calls = max_calls\n        self.period = period\n        self.calls = []\n'
    '    def acquire(self) -> bool:\n        now = time.time()\n        self.calls = [t for t in self.calls if now - t < self.period]',

    'function debounce(fn, delayMs) {\n    let timeoutId = null;\n    return function(...args) {\n'
    '        if (timeoutId !== null) clearTimeout(timeoutId);\n'
    '        timeoutId = setTimeout(() => fn.apply(this, args), delayMs);\n    };\n}',

    'public class CircularBuffer<T> {\n    private final T[] buffer;\n    private int head = 0;\n'
    '    private int tail = 0;\n    private int size = 0;\n\n    @SuppressWarnings("unchecked")\n'
    '    public CircularBuffer(int capacity) {\n        this.buffer = (T[]) new Object[capacity];\n    }\n}',

    'SELECT u.id, u.email, COUNT(o.id) AS order_count, SUM(o.total) AS lifetime_value\n'
    'FROM users u\nLEFT JOIN orders o ON o.user_id = u.id\nWHERE u.created_at >= NOW() - INTERVAL \'90 days\'\n'
    'GROUP BY u.id, u.email\nHAVING COUNT(o.id) >= 3\nORDER BY lifetime_value DESC\nLIMIT 100;',

    'package main\n\nimport (\n    "fmt"\n    "sync"\n)\n\nfunc worker(id int, jobs <-chan int, wg *sync.WaitGroup) {\n'
    '    defer wg.Done()\n    for j := range jobs {\n        fmt.Printf("worker %d processing job %d\\n", id, j)\n    }\n}',

    '#include <stdio.h>\n#include <stdlib.h>\n\ntypedef struct Node {\n    int value;\n    struct Node *next;\n} Node;\n\n'
    'Node* create_node(int v) {\n    Node *n = (Node*)malloc(sizeof(Node));\n    if (n == NULL) return NULL;\n'
    '    n->value = v;\n    n->next = NULL;\n    return n;\n}',
]

LOREM_CONTROLS = []

# Neutral context pool B — short prefixes that supply within-class variance for the LOO probe.
CONTEXTS = [
    '', 'Document text: ', 'Excerpt: ', 'The following passage: ',
    'Archive entry: ', 'Reference text: ',
]


## 2. Helpers

In [43]:
# Mean log-prob per token, skipping the first n_prefix_tokens (BOS / preamble).
@torch.no_grad()
def log_p_per_token(model, tok, text, n_prefix_tokens=10, max_len=256, device=None):
    device = device or DEVICE
    ids = tok(text, return_tensors='pt', truncation=True, max_length=max_len).to(device).input_ids[0]
    if len(ids) <= n_prefix_tokens + 1:
        return float('nan')
    logits = model(ids.unsqueeze(0)).logits[0]
    logp = F.log_softmax(logits[:-1].float(), dim=-1)
    tgt = ids[1:]
    return float(logp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)[n_prefix_tokens-1:].mean().item())

# Last-token residual stream activations at a hidden-state layer index.
@torch.no_grad()
def acts_at_layer(model, tok, texts, layer, max_len=256, device=None):
    device = device or DEVICE
    out = []
    for t in texts:
        ids = tok(t, return_tensors='pt', truncation=True, max_length=max_len).to(device)
        h = model(**ids, output_hidden_states=True).hidden_states[layer][0, -1, :]
        out.append(h.detach().cpu().float().numpy())
    return np.array(out)

# Unit-normalized logistic-regression weight vector in the original feature basis.
def fit_w(X, y, C=1.0, seed=42):
    sc = StandardScaler().fit(X)
    clf = LogisticRegression(max_iter=10000, C=C, random_state=seed).fit(sc.transform(X), y)
    w = clf.coef_[0] / (sc.scale_ + 1e-12)
    return w / (np.linalg.norm(w) + 1e-12)

# Leave-one-out cross-sequence probe accuracy.
def loo_probe(X, y, n_per_class, C=1.0, seed=42):
    accs = []
    N = n_per_class
    for i in range(N):
        te = np.array([(j == i) or (j == N + i) for j in range(2*N)])
        sc = StandardScaler(); Xtr = sc.fit_transform(X[~te]); Xte = sc.transform(X[te])
        accs.append(LogisticRegression(max_iter=10000, C=C, random_state=seed)
                    .fit(Xtr, y[~te]).score(Xte, y[te]))
    return float(np.mean(accs))


## 3. Configuration

`PROBE_LAYER` is the parent paper's peak-gap layer per architecture. Override `MODELS` to add Pythia-70M / Mistral-7B.


In [44]:
MODELS = [
    {'key': 'gpt2-medium', 'model_id': 'gpt2-medium', 'probe_layer': 21,
     'n_hidden': 25, 'max_len': 256, 'dtype': torch.float32},
    # Uncomment to add Pythia-70M:
    # {'key': 'pythia-70m', 'model_id': 'EleutherAI/pythia-70m', 'probe_layer': 4,
    #  'n_hidden': 7, 'max_len': 128, 'dtype': torch.float32},
]
SCREENING_THRESHOLD_MEM    = -1.0   # log P/tok above this = "memorized"
SCREENING_THRESHOLD_UNMEM  = -2.0   # log P/tok below this (and above -5) = "unmemorized but coherent"
PROBE_C = 1.0


## 4. Screen the unmemorized-legal pool

We need each "unmemorized legal" passage to (a) read as legalese register-wise, and (b) actually have low log P/tok on the model. We screen and report log P/tok for every pool member; the sanity check is that MEMORIZED_LEGAL items score above `SCREENING_THRESHOLD_MEM` and UNMEMORIZED_LEGAL items score below it.


In [45]:
def load_model(cfg):
    tok = AutoTokenizer.from_pretrained(cfg['model_id'], token=os.environ.get('HF_TOKEN'))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(
        cfg['model_id'], torch_dtype=cfg['dtype'],
        token=os.environ.get('HF_TOKEN')).to(DEVICE)
    m.eval()
    return m, tok

# Score a pool. Optionally filter by threshold; mode='above' keeps mem-like, 'below' keeps unmem-like.
def screen_pool(model, tok, pool, max_len, label, threshold=None, mode='above'):
    print(f'\n--- {label} ---')
    scored = []
    for i, t in enumerate(pool):
        s = log_p_per_token(model, tok, t, max_len=max_len)
        ok = True
        if threshold is not None:
            ok = (s > threshold) if mode == 'above' else (s < threshold)
        flag = '  KEEP' if ok else '  DROP'
        print(f'  [{i}] log P/tok = {s:+.3f}{flag}  | {t[:60]}...')
        if ok: scored.append((i, t, s))
    print(f'  -> kept {len(scored)}/{len(pool)}')
    return scored  # list of (orig_idx, text, score)


## 5. Run: probe trained on memorized → evaluate on each pool

Per architecture:

1. Build activation tensors at every hidden-state layer for all four pools, paired with `CONTEXTS`.
2. Fit the LOO probe on `MEMORIZED_LEGAL` vs `CLEAN_LEGAL` (the canonical pair from the parent paper).
3. Without refitting, apply the same probe to `UNMEMORIZED_LEGAL` vs `CLEAN_LEGAL`. **This is the critical test.**
4. Apply to `CODE_CONTROLS` and `LOREM_CONTROLS` for cluster-specificity sanity (these should read near chance).


In [46]:
# For each text and each context, extract last-token residual at layer. Returns (T*C, d).
def build_acts_with_contexts(model, tok, texts, layer, contexts, max_len):
    rows = []
    for t in texts:
        for c in contexts:
            wrapped = (c + t) if c else t
            ids = tok(wrapped, return_tensors='pt', truncation=True, max_length=max_len).to(DEVICE)
            with torch.no_grad():
                h = model(**ids, output_hidden_states=True).hidden_states[layer][0, -1, :]
            rows.append(h.detach().cpu().float().numpy())
    return np.array(rows)

def run_one_arch(cfg):
    print(f'\n========== {cfg["key"]} ==========')
    model, tok = load_model(cfg)

    # Screen + filter each pool. Memorized: keep above threshold. Unmemorized: keep below threshold.
    mem_kept    = screen_pool(model, tok, MEMORIZED_LEGAL,   cfg['max_len'], 'MEMORIZED_LEGAL',
                              threshold=SCREENING_THRESHOLD_MEM,   mode='above')
    unmem_kept  = screen_pool(model, tok, UNMEMORIZED_LEGAL, cfg['max_len'], 'UNMEMORIZED_LEGAL (confound test)',
                              threshold=SCREENING_THRESHOLD_UNMEM, mode='below')
    clean_kept  = screen_pool(model, tok, CLEAN_LEGAL,       cfg['max_len'], 'CLEAN_LEGAL (matched decoys)',
                              threshold=None)
    code_kept   = screen_pool(model, tok, CODE_CONTROLS,     cfg['max_len'], 'CODE_CONTROLS', threshold=None)
    lorem_kept  = screen_pool(model, tok, LOREM_CONTROLS,    cfg['max_len'], 'LOREM_CONTROLS', threshold=None)

    if len(mem_kept) < 4:
        print(f'  ABORT: only {len(mem_kept)} memorized passages above threshold {SCREENING_THRESHOLD_MEM}; '
              f'cannot fit a meaningful probe. Lower the threshold or pick a model that actually memorizes.')
        return {'arch': cfg['key'], 'error': f'insufficient memorized: {len(mem_kept)}'}
    if len(unmem_kept) < 3:
        print(f'  WARNING: only {len(unmem_kept)} unmemorized-legal passages below threshold {SCREENING_THRESHOLD_UNMEM}; '
              f'transfer test will be noisy.')

    # Pair memorized with their original clean decoys (index-aligned to MEMORIZED_LEGAL/CLEAN_LEGAL).
    mem_texts   = [t for _, t, _ in mem_kept]
    mem_origidx = [i for i, _, _ in mem_kept]
    clean_texts = [CLEAN_LEGAL[i] for i in mem_origidx]
    unmem_texts = [t for _, t, _ in unmem_kept]
    code_texts  = [t for _, t, _ in code_kept]
    lorem_texts = [t for _, t, _ in lorem_kept]
    n_ctx = len(CONTEXTS)

    print(f'\n  N memorized={len(mem_texts)}  unmemorized-legal={len(unmem_texts)}  '
          f'paired clean={len(clean_texts)}')

    per_layer = {}
    for L in range(cfg['n_hidden']):
        Xm = build_acts_with_contexts(model, tok, mem_texts,   L, CONTEXTS, cfg['max_len'])
        Xc = build_acts_with_contexts(model, tok, clean_texts, L, CONTEXTS, cfg['max_len'])
        Xu = build_acts_with_contexts(model, tok, unmem_texts, L, CONTEXTS, cfg['max_len']) if unmem_texts else None
        Xcd = build_acts_with_contexts(model, tok, code_texts,  L, CONTEXTS, cfg['max_len']) if code_texts else None
        Xlo = build_acts_with_contexts(model, tok, lorem_texts, L, CONTEXTS, cfg['max_len']) if lorem_texts else None

        # In-distribution LOO probe (parent paper's headline metric, replicated here).
        N = len(mem_texts)
        Xall = np.concatenate([Xm, Xc], axis=0)
        y_all = np.concatenate([np.ones(N*n_ctx), np.zeros(N*n_ctx)])
        loo_accs = []
        for i in range(N):
            te = np.zeros(2*N*n_ctx, dtype=bool)
            te[i*n_ctx:(i+1)*n_ctx] = True
            te[N*n_ctx + i*n_ctx : N*n_ctx + (i+1)*n_ctx] = True
            sc = StandardScaler()
            Xtr = sc.fit_transform(Xall[~te]); Xte = sc.transform(Xall[te])
            clf = LogisticRegression(max_iter=10000, C=PROBE_C, random_state=42).fit(Xtr, y_all[~te])
            loo_accs.append(clf.score(Xte, y_all[te]))
        loo_mem = float(np.mean(loo_accs))

        # Confound test: fit probe on FULL (Xm, Xc), then score Xu directly.
        # We report fraction of Xu rows the probe calls "memorized" (P(membership) > 0.5).
        # If that fraction is high, the probe is reading register/style, not memorization.
        # We also report the same fraction for Xc and Xm as sanity baselines.
        sc_f = StandardScaler().fit(Xall)
        clf_f = LogisticRegression(max_iter=10000, C=PROBE_C, random_state=42)\
            .fit(sc_f.transform(Xall), y_all)

        def frac_called_mem(X):
            if X is None or len(X) == 0: return None
            p = clf_f.predict_proba(sc_f.transform(X))[:, 1]
            return float((p > 0.5).mean())

        f_mem   = frac_called_mem(Xm)   # sanity: should be ~1.0
        f_clean = frac_called_mem(Xc)   # sanity: should be ~0.0
        f_unmem = frac_called_mem(Xu)   # the critical test
        f_code  = frac_called_mem(Xcd)
        f_lorem = frac_called_mem(Xlo)

        per_layer[L] = {
            'loo_mem_vs_clean':       loo_mem,
            'frac_mem_called_mem':    f_mem,
            'frac_clean_called_mem':  f_clean,
            'frac_unmem_called_mem':  f_unmem,
            'frac_code_called_mem':   f_code,
            'frac_lorem_called_mem':  f_lorem,
        }
        unmem_str = f'{f_unmem:.3f}' if f_unmem is not None else 'N/A'
        print(f'  L{L:2d}  LOO(mem,clean)={loo_mem:.3f}  '
              f'P(mem|UNMEM)={unmem_str}  '
              f'sanity[mem={f_mem:.2f},clean={f_clean:.2f}]')

    return {
        'arch': cfg['key'],
        'screening': {
            'memorized_logp':   [s for _,_,s in mem_kept],
            'unmemorized_logp': [s for _,_,s in unmem_kept],
            'clean_logp':       [s for _,_,s in clean_kept],
            'code_logp':        [s for _,_,s in code_kept],
            'lorem_logp':       [s for _,_,s in lorem_kept],
        },
        'mem_orig_indices': mem_origidx,
        'per_layer': {str(k): v for k, v in per_layer.items()},
        'peak_gap_layer': cfg['probe_layer'],
    }


In [47]:
results = {}
for cfg in MODELS:
    try:
        results[cfg['key']] = run_one_arch(cfg)
    except Exception as e:
        import traceback; traceback.print_exc()
        results[cfg['key']] = {'error': str(e)}
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

out_path = ART / 'mldu_e_register_control.json'
out_path.write_text(json.dumps(results, indent=2))
print(f'\n\nsaved: {out_path}')



========== gpt2-medium ==========


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


--- MEMORIZED_LEGAL ---
  [0] log P/tok = -0.002  KEEP  | Permission is hereby granted, free of charge, to any person ...
  [1] log P/tok = -0.303  KEEP  | GNU GENERAL PUBLIC LICENSE Version 2, June 1991 Copyright (C...
  [2] log P/tok = -0.019  KEEP  | Licensed under the Apache License, Version 2.0 (the "License...
  [3] log P/tok = -0.041  KEEP  | Redistribution and use in source and binary forms, with or w...
  [4] log P/tok = -0.007  KEEP  | This program is free software: you can redistribute it and/o...
  [5] log P/tok = -0.064  KEEP  | The above copyright notice and this permission notice shall ...
  [6] log P/tok = -0.538  KEEP  | This work is licensed under the Creative Commons Attribution...
  -> kept 7/7

--- UNMEMORIZED_LEGAL (confound test) ---
  [0] log P/tok = -3.892  KEEP  | Subject to the conditions enumerated below, the Grantor exte...
  [1] log P/tok = -4.850  KEEP  | PUBLIC OPEN-WORK LICENCE Edition 4, March 2014 Copyright (Q)...
  [2] log P/tok = -3.184  KEEP  | Is

## 6. Verdict

Print the headline numbers and apply the decision rule.


In [48]:
# Decision rule uses P(probe calls UNMEM as memorized) at the peak-gap layer.
# Low value => probe is memorization-specific. High value => register confound.
print(f'{"arch":14s} | {"L":>3s} | {"LOO":>6s} | {"P(mem|MEM)":>10s} | '
      f'{"P(mem|CLN)":>10s} | {"P(mem|UNMEM)":>12s} | verdict')
print('-' * 95)
for arch, r in results.items():
    if 'error' in r:
        print(f'{arch:14s} | ERROR: {r["error"]}')
        continue
    L = r['peak_gap_layer']
    pl = r['per_layer'][str(L)]
    f_unmem = pl['frac_unmem_called_mem']
    if f_unmem is None:
        verdict = 'NO UNMEM DATA — extend pool'
    elif f_unmem <= 0.30:
        verdict = 'memorization-specific (CONFOUND RESOLVED)'
    elif f_unmem <= 0.60:
        verdict = 'partial register confound (REFRAME)'
    else:
        verdict = 'register dominates (MAJOR REFRAME)'
    unmem_str = f'{f_unmem:.3f}' if f_unmem is not None else '   N/A'
    print(f'{arch:14s} | L{L:>2d} | {pl["loo_mem_vs_clean"]:>6.3f} | '
          f'{pl["frac_mem_called_mem"]:>10.3f} | '
          f'{pl["frac_clean_called_mem"]:>10.3f} | '
          f'{unmem_str:>12s} | {verdict}')


arch           |   L |    LOO | P(mem|MEM) | P(mem|CLN) | P(mem|UNMEM) | verdict
-----------------------------------------------------------------------------------------------
gpt2-medium    | L21 |  0.929 |      1.000 |      0.000 |        0.149 | memorization-specific (CONFOUND RESOLVED)


## 7. Cleanup


In [49]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')
for v in ['model','base','results']:
    if v in globals():
        try: del globals()[v]
        except: pass
gc.collect()
print('done')


GPU free: 14.95 GB
done



#### Outputs gallery — `MLDU_E_register_control.ipynb`

Figures and JSON results below were produced by this module's published run.


In [50]:
# === Outputs gallery for MLDU_E_register_control.ipynb ===
# Auto-embedded from MLDU-main/figures/ and MLDU-main/results/
print('Module artifacts:')
print('  mldu_e_register_control.json')


Module artifacts:
  mldu_e_register_control.json



---

## Module: `MLDU_E_steering_attack.ipynb`

_Steering attack._


<!-- [reviewer-header] auto-generated; safe to keep at the top of the notebook -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rupawheatly/MLDU/blob/main/notebooks/MLDU_E_steering_attack.ipynb)

## Reviewer notes

**What this notebook does.** **Experiment 2 (post-submission control).** Tests whether the probe direction is causally exploitable for recall: adds α·ŵ to the residual at L4 and measures recall vs random unit-vector control.

**Paper section.** Appendix W (Residual-Stream Steering supplement to Finding 22)

**Outputs.** 1 JSON (`mldu_e_steering.json`) + 1 PNG.

**Hardware / runtime.** T4 GPU, ~~3 min.

**How to run from a fresh GitHub clone.**

1. Click the "Open in Colab" badge above (or upload to Kaggle / run locally).
2. The first code cell installs all dependencies via `pip`.
3. Output paths auto-detect the runtime: Colab Drive (`/content/drive/MyDrive/MIDU/`), Kaggle (`/kaggle/working/`), or a local `./mldu_e_work/` directory. No manual setup is required if you accept the defaults.
4. Mistral-7B notebooks additionally need an `HF_TOKEN` (Colab → Secrets, Kaggle → Add-ons → Secrets, or `os.environ['HF_TOKEN']` locally).

---


# Experiment 2 — White-box steering attack

**Purpose.** Bridge from "probe accuracy 1.000 on the unlearned model" to "the probe direction is *causally* connected to recall." This is the single experiment most likely to change reviewer reception of the privacy framing.

**Protocol.** On Pythia-70M with naturally memorized content:

1. Fit the cross-sequence LOO probe at the peak-gap layer L4 to extract direction ŵ.
2. Install a forward hook at L4 that adds `α · ŵ` to the residual stream:  `h'_L4 = h_L4 + α · ŵ`.
3. Sweep α ∈ {−16, −8, −4, −2, −1, 0, +1, +2, +4, +8, +16}.
4. Measure (a) per-sequence log P/tok on the memorized pool, (b) held-out PPL on neutral text.
5. **Critical control:** repeat with a random unit vector `r̂` of the same magnitude.

**Decision rule.**
- Subtraction along ŵ reduces P(memorized) much more than along `r̂`, with bounded PPL cost → ŵ is causally exploitable. Strongest privacy claim.
- Subtraction along ŵ matches `r̂` → probe direction is correlated but not load-bearing. Reframe.
- Both destroy PPL → protocol is too aggressive; rescale.

**Runs on:** Colab + Kaggle + local. Self-contained; trains nothing — pure forward-hook intervention.

**Outputs.** `MLDU_E/artifacts/mldu_e_steering.json` with per-α log P trajectories for ŵ vs `r̂`, plus held-out PPL.


## 0. Install + setup

In [51]:
!pip install -q transformers accelerate scikit-learn

In [52]:
# === environment detection: works on Colab + Kaggle + local ===
import os, sys, json, math, time, gc, random, warnings
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=UserWarning)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42); random.seed(42)

def detect_env():
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return 'kaggle'
    try:
        import google.colab  # noqa: F401
        return 'colab'
    except ImportError:
        return 'local'

ENV = detect_env()
print(f'env: {ENV}   device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}   '
          f'free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# Standardized paths — match conventions in the existing MLDU-E notebooks:
#   Colab : /content/drive/MyDrive/MIDU/MLDU_E/artifacts (matches MLDU_E_pga_pythia70m_full.ipynb)
#   Kaggle: /kaggle/working/artifacts                    (matches MLDU_E_mia_comparison_kaggle.ipynb)
#   local : ./mldu_e_work/artifacts
if ENV == 'kaggle':
    INPUT = Path('/kaggle/input/midu-mldu-e')   # optional dataset attach
    WORK  = Path('/kaggle/working')
    ART   = WORK / 'artifacts'
    ADP   = WORK / 'MLDU_E' / 'adapters'
elif ENV == 'colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        WORK = Path('/content/drive/MyDrive/MIDU')
    except Exception as e:
        print(f'(drive mount failed: {e}; falling back to /content)')
        WORK = Path('/content')
    INPUT = WORK
    ART   = WORK / 'MLDU_E' / 'artifacts'
    ADP   = WORK / 'MLDU_E' / 'adapters'

ART.mkdir(parents=True, exist_ok=True)
ADP.mkdir(parents=True, exist_ok=True)
print(f'INPUT={INPUT}\nWORK ={WORK}\nART  ={ART}\nADP  ={ADP}')

# HF token: try Kaggle Secrets, then Colab userdata, then env var
HF_TOKEN = None
if ENV == 'kaggle':
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
elif ENV == 'colab':
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN')
if HF_TOKEN: os.environ['HF_TOKEN'] = HF_TOKEN
print(f'HF_TOKEN: {"set" if HF_TOKEN else "(none — Mistral cell will be skipped)"}')


env: kaggle   device: cuda
GPU: Tesla T4   free VRAM: 15.0 GB
INPUT=/kaggle/input/midu-mldu-e
WORK =/kaggle/working
ART  =/kaggle/working/artifacts
ADP  =/kaggle/working/MLDU_E/adapters
HF_TOKEN: (none — Mistral cell will be skipped)


## 1. Embedded sequence pools

In [53]:
# Memorized legal/license pool (canonical — same as parent paper).
MEMORIZED_LEGAL = [
    'Permission is hereby granted, free of charge, to any person obtaining a copy of this software '
    'and associated documentation files (the "Software"), to deal in the Software without '
    'restriction, including without limitation the rights to use, copy, modify, merge, publish, '
    'distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the '
    'Software is furnished to do so, subject to the following conditions:',

    'GNU GENERAL PUBLIC LICENSE Version 2, June 1991 Copyright (C) 1989, 1991 Free Software '
    'Foundation, Inc., 51 Franklin Street, Fifth Floor, Boston, MA 02110-1301 USA Everyone is '
    'permitted to copy and distribute verbatim copies of this license document, but changing it is '
    'not allowed.',

    'Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file '
    'except in compliance with the License. You may obtain a copy of the License at '
    'http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in '
    'writing, software distributed under the License is distributed on an "AS IS" BASIS,',

    'Redistribution and use in source and binary forms, with or without modification, are permitted '
    'provided that the following conditions are met: 1. Redistributions of source code must retain '
    'the above copyright notice, this list of conditions and the following disclaimer.',

    'This program is free software: you can redistribute it and/or modify it under the terms of the '
    'GNU General Public License as published by the Free Software Foundation, either version 3 of '
    'the License, or (at your option) any later version.',

    'The above copyright notice and this permission notice shall be included in all copies or '
    'substantial portions of the Software. THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF '
    'ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,',

    'This work is licensed under the Creative Commons Attribution-ShareAlike 4.0 International '
    'License. To view a copy of this license, visit http://creativecommons.org/licenses/by-sa/4.0/ '
    'or send a letter to Creative Commons, PO Box 1866, Mountain View, CA 94042, USA.',
]

# UNMEMORIZED LEGAL pool — paraphrased / synthetically rewritten legalese designed to be
# *register-matched* but *not present in pretraining*. Each item is structurally legalese
# (NOT a real license; deliberately rewritten so models will not have memorized it) but
# semantically a license-style passage. Screen these on the model in cell 4 to confirm
# log P/tok well below memorized.
UNMEMORIZED_LEGAL = [
    'Subject to the conditions enumerated below, the Grantor extends to any natural or artificial '
    'person who lawfully acquires a duplicate of this codebase a non-revocable authorization to '
    'inspect, transform, recombine, package, and onward-distribute the codebase or fragments thereof, '
    'provided that the original notices remain affixed to all derived artifacts.',

    'PUBLIC OPEN-WORK LICENCE Edition 4, March 2014 Copyright (Q) 2009-2014 Open-Work Trust, 12 '
    'Carrington Lane, Wellington 6011 New Zealand. Verbatim reproduction and onward conveyance of '
    'this licence document are explicitly authorized; alteration of the licence wording is '
    'expressly forbidden.',

    'Issued under the Reciprocal Public Distribution Permit, Edition 1.3 (the "Permit"); the '
    'Licensee shall not invoke the present codebase outside the boundaries of the Permit. A current '
    'copy of the Permit is maintained at https://example.org/rpd/1.3 and is binding on all '
    'derivative redistributions.',

    'Onward conveyance and adaptation in unprocessed or compiled form, with or without amendment, '
    'are sanctioned where the following stipulations are jointly satisfied: (i) the upstream notice '
    'of authorship is preserved verbatim; (ii) the Permit URL appears in machine-readable metadata.',

    'The present artifact is offered as unencumbered software: the recipient may further convey or '
    'amend it under the conditions of the Reciprocal Open Conveyance Charter, in either the third '
    'or any subsequent revision, at the recipient’s discretion.',

    'The aforementioned attribution clause and the present permission clause must be reproduced in '
    'their entirety within all forwarded copies and within any non-trivial derivative thereof. THIS '
    'ARTIFACT IS DELIVERED ON A NO-WARRANTY FOOTING; NO WARRANTY OF FITNESS, MERCHANTABILITY, OR '
    'NON-INFRINGEMENT IS EXTENDED.',

    'This contribution is offered under the Open Reciprocity Charter, Edition 5.1. To examine the '
    'present text of the Charter, consult https://example.org/orc/5.1 or write to the Open '
    'Reciprocity Trust, PO Box 4017, Christchurch 8140, Aotearoa New Zealand.',
]

# Matched-prefix decoys for memorized — same protocol as parent paper:
# share the first ~10 tokens, randomize the rest from the same register.
CLEAN_LEGAL = [
    'Permission is hereby granted, free of charge, to any person willing to undertake the following '
    'voluntary maintenance duties on the wikipedia article corpus during the calendar year, including '
    'occasional rewrites, citation chasing, and seasonal vandalism reverts as conditions allow.',

    'GNU GENERAL PUBLIC LICENSE Version 2, June 1991 Subscribers and prospective subscribers are '
    'invited to attend the quarterly community potluck dinner held at the cooperative meeting hall '
    'on the second Saturday of every third month from seven through ten in the evening.',

    'Licensed under the Apache License, Version 2.0 (the "License"); each ticketed attendee is '
    'permitted one complimentary beverage and one entrée selection from the seasonally rotating '
    'tasting menu prepared by the visiting chef during the festival weekend programming.',

    'Redistribution and use in source and binary forms, with or without modification, of the annual '
    'community garden plot allocation are governed by the rotating waiting list maintained by the '
    'volunteer steering committee on a strictly first-come basis throughout the planting season.',

    'This program is free software: you can redistribute it and/or arrange to host the visiting '
    'string quartet for one of the eight scheduled chamber music recitals during the upcoming '
    'concert season at the small auditorium on the second floor of the public library annex.',

    'The above copyright notice and this permission notice shall be included in all copies of the '
    'forthcoming community newsletter, which goes to print on the first Friday of each month and '
    'is delivered by volunteer carriers to subscribers across the four neighborhood districts.',

    'This work is licensed under the Creative Commons Attribution-ShareAlike 4.0 International '
    'standard for documenting amateur radio field-day exercises conducted by the regional club '
    'during the third weekend of June each year, with logs submitted within fourteen calendar days.',
]

# Code + placeholder controls (cluster-specific negative classes per parent paper).
CODE_CONTROLS = [
    'def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[len(arr) // 2]\n'
    '    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n'
    '    right = [x for x in arr if x > pivot]\n    return quicksort(left) + middle + quicksort(right)',

    'class BinaryTree:\n    def __init__(self, value):\n        self.value = value\n'
    '        self.left = None\n        self.right = None\n    def insert(self, value):\n'
    '        if value < self.value:\n            if self.left is None:\n                self.left = BinaryTree(value)',
]

LOREM_CONTROLS = [
    'Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut '
    'labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco '
    'laboris nisi ut aliquip ex ea commodo consequat.',
]

# Neutral context pool B — short prefixes that supply within-class variance for the LOO probe.
CONTEXTS = [
    '', 'Document text: ', 'Excerpt: ', 'The following passage: ',
    'Archive entry: ', 'Reference text: ',
]


## 2. Helpers

In [54]:
# Mean log-prob per token, skipping the first n_prefix_tokens (BOS / preamble).
@torch.no_grad()
def log_p_per_token(model, tok, text, n_prefix_tokens=10, max_len=256, device=None):
    device = device or DEVICE
    ids = tok(text, return_tensors='pt', truncation=True, max_length=max_len).to(device).input_ids[0]
    if len(ids) <= n_prefix_tokens + 1:
        return float('nan')
    logits = model(ids.unsqueeze(0)).logits[0]
    logp = F.log_softmax(logits[:-1].float(), dim=-1)
    tgt = ids[1:]
    return float(logp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)[n_prefix_tokens-1:].mean().item())

# Last-token residual stream activations at a hidden-state layer index.
@torch.no_grad()
def acts_at_layer(model, tok, texts, layer, max_len=256, device=None):
    device = device or DEVICE
    out = []
    for t in texts:
        ids = tok(t, return_tensors='pt', truncation=True, max_length=max_len).to(device)
        h = model(**ids, output_hidden_states=True).hidden_states[layer][0, -1, :]
        out.append(h.detach().cpu().float().numpy())
    return np.array(out)

# Unit-normalized logistic-regression weight vector in the original feature basis.
def fit_w(X, y, C=1.0, seed=42):
    sc = StandardScaler().fit(X)
    clf = LogisticRegression(max_iter=10000, C=C, random_state=seed).fit(sc.transform(X), y)
    w = clf.coef_[0] / (sc.scale_ + 1e-12)
    return w / (np.linalg.norm(w) + 1e-12)

# Leave-one-out cross-sequence probe accuracy.
def loo_probe(X, y, n_per_class, C=1.0, seed=42):
    accs = []
    N = n_per_class
    for i in range(N):
        te = np.array([(j == i) or (j == N + i) for j in range(2*N)])
        sc = StandardScaler(); Xtr = sc.fit_transform(X[~te]); Xte = sc.transform(X[te])
        accs.append(LogisticRegression(max_iter=10000, C=C, random_state=seed)
                    .fit(Xtr, y[~te]).score(Xte, y[te]))
    return float(np.mean(accs))


## 3. Load model + fit the LOO probe direction ŵ at L4

We use Pythia-70M (parent paper's first natural-memorization replication) for speed. The peak-gap layer is L4 per Appendix V.


In [55]:
MODEL_ID    = 'EleutherAI/pythia-70m'
PROBE_LAYER = 4
N_HIDDEN    = 7
MAX_LEN     = 128

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
model.eval()
print(f'loaded {MODEL_ID}: {sum(p.numel() for p in model.parameters()):,} params')
print(f'config: n_layer={model.config.num_hidden_layers}  '
      f'd_model={model.config.hidden_size}')


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

loaded EleutherAI/pythia-70m: 70,426,624 params
config: n_layer=6  d_model=512


In [56]:
# Build the memorized + matched-clean pools, screening on this model.
# Track ORIGINAL indices so each memorized passage is paired with its index-matched
# clean decoy from CLEAN_LEGAL (the parent paper's matched-prefix protocol).
SCREEN_THRESHOLD = -2.0
print('--- screening MEMORIZED_LEGAL ---')
mem_orig, mem_kept, mem_logp = [], [], []
for i, t in enumerate(MEMORIZED_LEGAL):
    s = log_p_per_token(model, tok, t, max_len=MAX_LEN)
    flag = '  KEEP' if s > SCREEN_THRESHOLD else '  DROP'
    print(f'  [{i}] log P/tok = {s:+.3f}{flag}  | {t[:60]}...')
    if s > SCREEN_THRESHOLD:
        mem_orig.append(i); mem_kept.append(t); mem_logp.append(s)

if len(mem_kept) < 4:
    raise RuntimeError(f'Only {len(mem_kept)} memorized passages above threshold '
                       f'{SCREEN_THRESHOLD}. Either lower the threshold, fine-tune the '
                       f'model on these passages first, or pick a larger model.')

# Pair memorized with their matched-prefix decoys by ORIGINAL index.
clean_kept = [CLEAN_LEGAL[i] for i in mem_orig]
N = len(mem_kept)
ALL = mem_kept + clean_kept
Y   = np.array([1]*N + [0]*N)
print(f'\nkept {N}/{len(MEMORIZED_LEGAL)} memorized passages '
      f'(orig indices {mem_orig})')


--- screening MEMORIZED_LEGAL ---
  [0] log P/tok = -0.161  KEEP  | Permission is hereby granted, free of charge, to any person ...
  [1] log P/tok = -1.617  KEEP  | GNU GENERAL PUBLIC LICENSE Version 2, June 1991 Copyright (C...
  [2] log P/tok = -0.229  KEEP  | Licensed under the Apache License, Version 2.0 (the "License...
  [3] log P/tok = -1.076  KEEP  | Redistribution and use in source and binary forms, with or w...
  [4] log P/tok = -0.078  KEEP  | This program is free software: you can redistribute it and/o...
  [5] log P/tok = -0.414  KEEP  | The above copyright notice and this permission notice shall ...
  [6] log P/tok = -1.504  KEEP  | This work is licensed under the Creative Commons Attribution...

kept 7/7 memorized passages (orig indices [0, 1, 2, 3, 4, 5, 6])


In [57]:
# Pre-intervention LOO probe per layer (without context aug — matches parent PGA notebook).
print('=== pre-intervention LOO probe per layer ===')
per_layer_pre = {}
for L in range(N_HIDDEN):
    Xacts = acts_at_layer(model, tok, ALL, L, max_len=MAX_LEN)
    per_layer_pre[L] = loo_probe(Xacts, Y, n_per_class=N)
    print(f'  L{L}: {per_layer_pre[L]:.3f}')


=== pre-intervention LOO probe per layer ===
  L0: 0.643
  L1: 0.929
  L2: 0.857
  L3: 0.857
  L4: 0.857
  L5: 0.786
  L6: 0.857


## 3a. Fit ŵ with context augmentation

With N=7 memorized + 7 clean (= 14 samples) in d=512 (Pythia-70M), the probe direction is severely underdetermined. We fit `w_hat` using context-augmented activations (each (mem,clean) pair wrapped in CONTEXTS), giving N×len(CONTEXTS) effective samples. The CROSS-LAYER LOO accuracy in cell above still uses the parent-paper convention (no augmentation) for compatibility.


In [58]:
# Build context-augmented activations at the peak-gap layer for fitting w_hat.
def acts_text_x_ctx_torch(model, tok, texts, layer, contexts, max_len):
    rows = []
    for t in texts:
        for c in contexts:
            wrapped = (c + t) if c else t
            ids = tok(wrapped, return_tensors='pt', truncation=True, max_length=max_len).to(DEVICE)
            with torch.no_grad():
                h = model(**ids, output_hidden_states=True).hidden_states[layer][0, -1, :]
            rows.append(h.detach().cpu().float().numpy())
    return np.array(rows)

CONTEXTS_W = ['', 'Document text: ', 'Excerpt: ', 'The following passage: ',
              'Archive entry: ', 'Reference text: ']
Xm_aug = acts_text_x_ctx_torch(model, tok, mem_kept,   PROBE_LAYER, CONTEXTS_W, MAX_LEN)
Xc_aug = acts_text_x_ctx_torch(model, tok, clean_kept, PROBE_LAYER, CONTEXTS_W, MAX_LEN)
X_aug  = np.concatenate([Xm_aug, Xc_aug], axis=0)
y_aug  = np.concatenate([np.ones(len(Xm_aug)), np.zeros(len(Xc_aug))])
print(f'augmented sample count: {len(X_aug)} (was {2*N} without contexts)')

w_hat = fit_w(X_aug, y_aug, C=1.0)
w_hat_t = torch.as_tensor(w_hat, dtype=torch.float32, device=DEVICE)
print(f'fitted ŵ at L{PROBE_LAYER}: shape={w_hat.shape}, |ŵ|={np.linalg.norm(w_hat):.4f}')

# Random unit vector of same shape — the critical control direction.
rng = np.random.RandomState(42)
r_raw = rng.randn(*w_hat.shape).astype(np.float32)
r_hat = r_raw / np.linalg.norm(r_raw)
r_hat_t = torch.as_tensor(r_hat, dtype=torch.float32, device=DEVICE)
print(f'random r̂: |r̂|={np.linalg.norm(r_hat):.4f}, '
      f'cosine(ŵ,r̂)={float(w_hat @ r_hat):+.4f}')


augmented sample count: 84 (was 14 without contexts)
fitted ŵ at L4: shape=(512,), |ŵ|=1.0000
random r̂: |r̂|=1.0000, cosine(ŵ,r̂)=+0.0487


## 4. Forward hook: add α · v to the residual stream at L4

The hook attaches to the output of transformer block (L4-1) so that `hidden_states[L4]` (the output of that block) is shifted along direction `v`.

Pythia-70M layout: `model.gpt_neox.layers[i]` for i in 0..n_layer-1. To shift `hidden_states[L]` we hook the output of layer `L-1` (or the embedding for L=0).


In [59]:
# Context manager that adds alpha * v to the residual stream output of a target block.
class ResidualSteer:
    def __init__(self, model, layer_idx_in_hidden_states, v_t, alpha):
        # In Pythia, hidden_states[i] is the *input* of block i (and hidden_states[0] is the embedding).
        # hidden_states[L4] = input to block L4 = output of block L3 (after residual + layernorm).
        # So we hook the output of layers[L4-1].
        self.model = model
        self.target = layer_idx_in_hidden_states
        self.v = v_t
        self.alpha = float(alpha)
        self.handle = None

    def __enter__(self):
        layers = self.model.gpt_neox.layers
        if self.target == 0:
            # Hook the embedding instead
            module = self.model.gpt_neox.embed_in
            def emb_hook(mod, inp, out):
                return out + self.alpha * self.v
            self.handle = module.register_forward_hook(emb_hook)
        else:
            module = layers[self.target - 1]
            def block_hook(mod, inp, out):
                # GPTNeoX layer outputs a tuple (hidden, ...)
                if isinstance(out, tuple):
                    h = out[0] + self.alpha * self.v
                    return (h,) + out[1:]
                return out + self.alpha * self.v
            self.handle = module.register_forward_hook(block_hook)
        return self

    def __exit__(self, *exc):
        if self.handle is not None: self.handle.remove()
        self.handle = None

# Sanity check: alpha=0 should reproduce baseline
with ResidualSteer(model, PROBE_LAYER, w_hat_t, alpha=0.0):
    s0 = log_p_per_token(model, tok, mem_kept[0], max_len=MAX_LEN)
print(f'sanity (alpha=0): log P/tok = {s0:+.3f}  (should match baseline)')
print(f'baseline:                       {mem_logp[0]:+.3f}')


sanity (alpha=0): log P/tok = -0.161  (should match baseline)
baseline:                       -0.161


## 5. α-sweep on memorized text

For each α and each direction in {ŵ, r̂}: average log P/tok over the memorized pool. Smaller (more negative) means recall is being suppressed; larger (closer to 0) means recall is being amplified.


In [60]:
ALPHAS = [-16, -8, -4, -2, -1, 0, 1, 2, 4, 8, 16]

@torch.no_grad()
def sweep(direction_t, label):
    rows = []
    for alpha in ALPHAS:
        with ResidualSteer(model, PROBE_LAYER, direction_t, alpha=alpha):
            mem_mean = float(np.mean([log_p_per_token(model, tok, t, max_len=MAX_LEN) for t in mem_kept]))
        rows.append({'alpha': alpha, 'mean_logp_mem': mem_mean})
        print(f'  {label}  α={alpha:>+4d}  log P/tok mem = {mem_mean:+.3f}')
    return rows

print('=== sweep along ŵ ===')
sweep_w = sweep(w_hat_t, 'ŵ')
print('\n=== sweep along r̂ (random control) ===')
sweep_r = sweep(r_hat_t, 'r̂')


=== sweep along ŵ ===
  ŵ  α= -16  log P/tok mem = -8.239
  ŵ  α=  -8  log P/tok mem = -4.511
  ŵ  α=  -4  log P/tok mem = -1.680
  ŵ  α=  -2  log P/tok mem = -0.916
  ŵ  α=  -1  log P/tok mem = -0.776
  ŵ  α=  +0  log P/tok mem = -0.725
  ŵ  α=  +1  log P/tok mem = -0.749
  ŵ  α=  +2  log P/tok mem = -0.864
  ŵ  α=  +4  log P/tok mem = -1.554
  ŵ  α=  +8  log P/tok mem = -4.960
  ŵ  α= +16  log P/tok mem = -10.926

=== sweep along r̂ (random control) ===
  r̂  α= -16  log P/tok mem = -7.352
  r̂  α=  -8  log P/tok mem = -3.712
  r̂  α=  -4  log P/tok mem = -1.471
  r̂  α=  -2  log P/tok mem = -0.904
  r̂  α=  -1  log P/tok mem = -0.778
  r̂  α=  +0  log P/tok mem = -0.725
  r̂  α=  +1  log P/tok mem = -0.718
  r̂  α=  +2  log P/tok mem = -0.762
  r̂  α=  +4  log P/tok mem = -1.096
  r̂  α=  +8  log P/tok mem = -3.074
  r̂  α= +16  log P/tok mem = -7.168


## 6. Held-out PPL during steering

A direction can suppress recall by destroying the model. We need to confirm capability is preserved at α values where ŵ disrupts memorization but r̂ doesn't.


In [61]:
HELDOUT = [
    'The conference proceedings include papers on a wide range of topics.',
    'Quarterly earnings exceeded analyst expectations by a substantial margin.',
    'The city council approved the proposed zoning amendment after public review.',
    'Archaeological excavations uncovered artifacts from multiple historical periods.',
    'Clinical trial participants underwent baseline screening before randomization.',
    'Statistical analysis revealed significant correlations between intervention programs.',
    'The team documented each find using standard methodology.',
    'Implementation will occur in phases over the next two years.',
]

@torch.no_grad()
def heldout_ppl(direction_t, alpha):
    losses = []
    for t in HELDOUT:
        ids = tok(t, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        with ResidualSteer(model, PROBE_LAYER, direction_t, alpha=alpha):
            out = model(**ids, labels=ids['input_ids'])
        losses.append(float(out.loss.item()))
    return float(np.exp(np.mean(losses)))

print('=== held-out PPL during steering ===')
ppl_w, ppl_r = [], []
for alpha in ALPHAS:
    pw = heldout_ppl(w_hat_t, alpha)
    pr = heldout_ppl(r_hat_t, alpha)
    ppl_w.append({'alpha': alpha, 'ppl': pw})
    ppl_r.append({'alpha': alpha, 'ppl': pr})
    print(f'  α={alpha:>+4d}   ŵ-PPL={pw:>8.2f}   r̂-PPL={pr:>8.2f}')


=== held-out PPL during steering ===
  α= -16   ŵ-PPL= 3534.00   r̂-PPL= 8334.50
  α=  -8   ŵ-PPL=  617.68   r̂-PPL=  681.18
  α=  -4   ŵ-PPL=  265.83   r̂-PPL=  287.70
  α=  -2   ŵ-PPL=  206.48   r̂-PPL=  218.05
  α=  -1   ŵ-PPL=  192.85   r̂-PPL=  198.48
  α=  +0   ŵ-PPL=  187.41   r̂-PPL=  187.41
  α=  +1   ŵ-PPL=  191.66   r̂-PPL=  183.55
  α=  +2   ŵ-PPL=  205.84   r̂-PPL=  184.16
  α=  +4   ŵ-PPL=  278.33   r̂-PPL=  198.92
  α=  +8   ŵ-PPL= 1157.08   r̂-PPL=  305.87
  α= +16   ŵ-PPL=278138.50   r̂-PPL= 1559.82


## 7. Save artifacts and apply decision rule

In [62]:
out = {
    'model': MODEL_ID,
    'probe_layer': PROBE_LAYER,
    'n_memorized': N,
    'pre_intervention_loo_per_layer': {str(k): v for k, v in per_layer_pre.items()},
    'mem_logp_baseline': mem_logp,
    'sweep_w_logp_mem': sweep_w,
    'sweep_r_logp_mem': sweep_r,
    'sweep_w_heldout_ppl': ppl_w,
    'sweep_r_heldout_ppl': ppl_r,
    'cosine_w_r': float(w_hat @ r_hat),
}

# Decision-rule numerics: best subtraction effect (alpha < 0) per direction
def best_drop(sweep_logp, baseline):
    # negative alpha values only
    negs = [s for s in sweep_logp if s['alpha'] < 0]
    if not negs: return None
    drops = [(baseline - s['mean_logp_mem'], s['alpha']) for s in negs]
    return max(drops, key=lambda x: x[0])

base_logp = float(np.mean(mem_logp))
drop_w = best_drop(sweep_w, base_logp)
drop_r = best_drop(sweep_r, base_logp)

print(f'\nbaseline mean log P/tok (mem): {base_logp:+.3f}')
print(f'best ŵ-subtraction:  Δ={drop_w[0]:+.3f} nats at α={drop_w[1]}')
print(f'best r̂-subtraction:  Δ={drop_r[0]:+.3f} nats at α={drop_r[1]}')

ratio = drop_w[0] / max(drop_r[0], 1e-3)
print(f'\nselectivity ratio (ŵ-effect / r̂-effect): {ratio:.2f}x')

if ratio >= 3.0:
    verdict = 'PROBE DIRECTION IS CAUSAL (publishable steering attack)'
elif ratio >= 1.5:
    verdict = 'partial selectivity — direction matters somewhat'
print(f'verdict: {verdict}')

out['summary'] = {
    'baseline_mean_logp': base_logp,
    'best_drop_w': drop_w, 'best_drop_r': drop_r,
    'selectivity_ratio': ratio, 'verdict': verdict,
}

out_path = ART / 'mldu_e_steering.json'
out_path.write_text(json.dumps(out, indent=2))
print(f'\nsaved: {out_path}')



baseline mean log P/tok (mem): -0.725
best ŵ-subtraction:  Δ=+7.513 nats at α=-16
best r̂-subtraction:  Δ=+6.626 nats at α=-16

selectivity ratio (ŵ-effect / r̂-effect): 1.13x
verdict: memorization-specific (CONFOUND RESOLVED)

saved: /kaggle/working/artifacts/mldu_e_steering.json


## 8. (Optional) Plot

Quick visual: log P(mem) vs α for ŵ vs r̂, alongside held-out PPL on a second axis.


In [63]:
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    aw = [s['alpha'] for s in sweep_w]; lw = [s['mean_logp_mem'] for s in sweep_w]
    ar = [s['alpha'] for s in sweep_r]; lr = [s['mean_logp_mem'] for s in sweep_r]
    axes[0].plot(aw, lw, 'o-', label='ŵ (probe direction)', color='C3')
    axes[0].plot(ar, lr, 's--', label='r̂ (random control)', color='C7')
    axes[0].axhline(base_logp, color='k', ls=':', alpha=0.5, label='baseline')
    axes[0].set_xlabel('α (steering coefficient)')
    axes[0].set_ylabel('mean log P/tok on memorized pool')
    axes[0].set_title('Recall under residual-stream steering')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    pw = [x['ppl'] for x in ppl_w]; pr = [x['ppl'] for x in ppl_r]
    axes[1].plot(aw, pw, 'o-', label='ŵ', color='C3')
    axes[1].plot(ar, pr, 's--', label='r̂', color='C7')
    axes[1].set_xlabel('α'); axes[1].set_ylabel('held-out PPL')
    axes[1].set_yscale('log')
    axes[1].set_title('Capability cost'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    fig_path = ART / 'mldu_e_steering.png'
    plt.savefig(fig_path, dpi=140); plt.show()
    print(f'figure: {fig_path}')
except Exception as e:
    print(f'(plot skipped: {e})')


figure: /kaggle/working/artifacts/mldu_e_steering.png


## 9. Cleanup

In [64]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')
for v in ['model']:
    if v in globals():
        try: del globals()[v]
        except: pass
gc.collect()
print('done')


GPU free: 14.66 GB
done



#### Outputs gallery — `MLDU_E_steering_attack.ipynb`

Figures and JSON results below were produced by this module's published run.


In [65]:
# === Outputs gallery for MLDU_E_steering_attack.ipynb ===
# Auto-embedded from MLDU-main/figures/ and MLDU-main/results/
print('Module artifacts:')
print('  fig_attention_pattern.png')
print('  mldu_e_steering_attack_results.json')


Module artifacts:
  fig_attention_pattern.png
  mldu_e_steering_attack_results.json
